# Qwen3-VL LoRA GRPO-Style RL on Robosuite + MuJoCo

This notebook is a dedicated RL variant derived from the local `robot_vlm_base.ipynb`.

Current training objective in this version:
- Simulation stays on standard `robosuite + mujoco`.
- `Qwen/Qwen3-VL-4B-Instruct` is loaded as the VLM backbone.
- RL updates **only LoRA adapters inside Qwen**.
- Training is planner-centric: Qwen generates natural-language / DSL plans, the environment executes them, and task completion provides reward.
- The update is GRPO-style in spirit: for the same task / scene seed, sample a group of candidate plans, compare their rewards, and update LoRA using group-relative advantages.

What this notebook does not try to do:
- No `MJWarp` integration.
- No MuJoCo GPU physics backend.
- No separate continuous-action policy head.
- No full dense finetuning of all Qwen parameters.

Why LoRA here:
- Full RL finetuning of a 4B VLM is not a practical notebook baseline.
- LoRA is the realistic way to let RL change Qwen behavior while keeping memory and optimizer state bounded.


---
## 1. System Dependencies

In [1]:
import subprocess, os

missing = any(
    subprocess.run(f"dpkg -l {pkg}".split(), capture_output=True).returncode != 0
    for pkg in ["libegl1-mesa", "libegl1-mesa-dev", "libglib2.0-0"]
)
if missing:
    !sudo apt-get update -qq && sudo apt-get install -y -qq libegl1-mesa libegl1-mesa-dev libglib2.0-0
    print("System deps installed.")
else:
    print("System deps already present.")

os.environ["MUJOCO_GL"] = "egl"
os.environ["EGL_DEVICE_ID"] = "0"

import os
import subprocess

# Headless rendering only. This does not change MuJoCo simulation to GPU physics.
os.environ.setdefault("MUJOCO_GL", "egl")

missing = any(
    subprocess.run(f"dpkg -l {pkg}".split(), capture_output=True).returncode != 0
    for pkg in ["libegl1-mesa", "libegl1-mesa-dev", "libglib2.0-0"]
)
if missing:
    raise RuntimeError(
        "Missing system packages. Install: libegl1-mesa libegl1-mesa-dev libglib2.0-0"
    )
print(f"MUJOCO_GL={os.environ['MUJOCO_GL']}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 10.)
debconf: falling back to frontend: Readline
Selecting previously unselected package libglx-dev:amd64.
(Reading database ... 125128 files and directories currently installed.)
Preparing to unpack .../0-libglx-dev_1.4.0-1_amd64.deb ...
Unpacking libglx-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgl-dev:amd64.
Preparing to unpack .../1-libgl-dev_1.4.0-1_amd64.deb ...
Unpacking libgl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libegl-dev:amd64.
Preparing to unpack .../2-libegl-dev_1.4.0-1_amd64.deb ...
Unpacking libegl-dev:amd64 (1.4.0-1) ...
Selectin

---
## 2. Python Packages

In [2]:
import importlib.machinery as _im
import os
import shutil
import subprocess
import sys

def _pip_install(pkg_spec):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkg_spec.split(),
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(r.stderr[-1000:])
        r.check_returncode()

_pip_install("numpy==2.0.2 scipy==1.14.1")
_pip_install("mujoco>=3.3.0 robosuite>=1.5.2")
_pip_install("torch torchvision")
_pip_install("transformers accelerate sentencepiece einops timm qwen-vl-utils")
_pip_install("peft bitsandbytes")
_pip_install("imageio imageio-ffmpeg pillow")

# Some environments probe for flash_attn even when it is not actually required.
_stub_dir = "/tmp/flash_attn"
os.makedirs(_stub_dir, exist_ok=True)
with open(f"{_stub_dir}/__init__.py", "w") as f:
    f.write('__version__ = "2.6.1"\n')
if "/tmp" not in sys.path:
    sys.path.insert(0, "/tmp")

# Keep numpy/scipy imports clean in environments with custom import hooks.
_clean = "/tmp/cleanpkgs"
shutil.rmtree(_clean, ignore_errors=True)
os.makedirs(_clean, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", _clean, "numpy==2.0.2", "scipy==1.14.1"],
    check=True,
    capture_output=True,
    text=True,
)
while _clean in sys.path:
    sys.path.remove(_clean)
sys.path.insert(0, _clean)
_orig_meta = sys.meta_path[:]
sys.meta_path = [_im.BuiltinImporter, _im.FrozenImporter, _im.PathFinder]
for _k in [k for k in list(sys.modules) if k == "numpy" or k.startswith("numpy.")]:
    del sys.modules[_k]
import numpy as np
import scipy
sys.meta_path = _orig_meta

import bitsandbytes
import flash_attn
import mujoco
import robosuite as suite
import torch
import transformers
import peft

print(f"numpy {np.__version__}")
print(f"scipy {scipy.__version__}")
print(f"mujoco {mujoco.__version__}")
print(f"robosuite {suite.__version__}")
print(f"torch {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"peft {peft.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print(f"flash_attn stub {flash_attn.__version__}")

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


numpy 2.0.2
scipy 1.14.1
mujoco 3.9.0
robosuite 1.5.2
torch 2.10.0+cu128
transformers 5.0.0
peft 0.18.1
bitsandbytes 0.49.2
flash_attn stub 2.6.1


---
## 3. Imports and Runtime Setup

In [3]:
import gc
import itertools
import logging
import os
import random
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import imageio.v2 as imageio
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from robosuite.controllers import load_composite_controller_config
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

for _name in list(logging.root.manager.loggerDict):
    if _name == "robosuite" or _name.startswith("robosuite."):
        _log = logging.getLogger(_name)
        _log.disabled = True
        _log.setLevel(logging.CRITICAL)
        _log.handlers.clear()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {DEVICE}")
print("MuJoCo simulation remains standard robosuite + mujoco regardless of CUDA availability.")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def to_tensor(x, device=DEVICE, dtype=torch.float32):
    return torch.as_tensor(x, dtype=dtype, device=device)

def atanh(x: torch.Tensor) -> torch.Tensor:
    x = x.clamp(-0.999999, 0.999999)
    return 0.5 * (torch.log1p(x) - torch.log1p(-x))

def count_trainable_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def count_all_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

Torch device: cuda
MuJoCo simulation remains standard robosuite + mujoco regardless of CUDA availability.


---
## 4. RL + LoRA Configuration

In [4]:
@dataclass
class RLConfig:
    vlm_model_id: str = "Qwen/Qwen3-VL-4B-Instruct"
    use_4bit: bool = True
    gradient_checkpointing: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    lora_target_modules: str = "all-linear"
    task_prompt: str = (
        "Pick up the red cube and place it into the wooden bin. "
        "The wooden bin is an open-top, four-walled container; its walls are about 10 cm (0.10 m) tall, "
        "so the bin rim sits roughly 10 cm above the table surface. "
        "Follow these steps in order:\n"
        "1. Move the gripper directly above the red cube and align it over the cube in the XY plane.\n"
        "2. Descend vertically until the gripper contacts the cube.\n"
        "3. Close the gripper to grasp the cube firmly.\n"
        "4. Lift the grasped cube straight UP by at least 0.15 m (use move_by offset [0,0,0.15] or more). "
        "This raises the cube HIGHER than the 10 cm bin wall - the cube MUST clear the wall before you "
        "move it sideways over the bin.\n"
        "5. Move the gripper over the wooden bin and hover directly above the bin's opening, still "
        "higher than the rim.\n"
        "6. Descend vertically so the cube drops down into the bin, below the rim.\n"
        "7. Open the gripper to release the cube inside the bin.\n"
        "The task is complete when the red cube rests inside the bin and the gripper has released it."
    )
    env_name: str = "Lift"
    robot: str = "Panda"
    camera_names: Tuple[str, str] = ("agentview", "robot0_eye_in_hand")
    camera_size: int = 256
    reward_shaping: bool = True
    control_freq: int = 20
    horizon: int = 200
    seed: int = 42
    action_repeat: int = 1
    hidden_dim: int = 1024  # unused in planner-only RL; kept for compatibility
    rollout_steps: int = 128  # unused in planner-only RL; kept for compatibility
    total_updates: int = 10
    ppo_epochs: int = 4  # unused in planner-only RL; kept for compatibility
    minibatch_size: int = 32  # unused in planner-only RL; kept for compatibility
    gamma: float = 0.99  # unused in planner-only RL; kept for compatibility
    gae_lambda: float = 0.95  # unused in planner-only RL; kept for compatibility
    clip_eps: float = 0.2  # unused in planner-only RL; kept for compatibility
    backbone_lr: float = 1e-4
    policy_lr: float = 3e-4  # unused in planner-only RL; kept for compatibility
    value_coef: float = 0.5  # unused in planner-only RL; kept for compatibility
    entropy_coef: float = 0.0  # unused in planner-only RL; kept for compatibility
    max_grad_norm: float = 1.0
    save_every: int = 10
    checkpoint_dir: str = "checkpoints_qwen_lora_rl"
    run_name: str = "qwen3_vl_lora_lift_ppo"
    video_fps: int = 10
    inference_task_type: str = "pick_and_place"
    inference_bbox_mode: str = "auto"
    inference_bbox_target: Optional[str] = None
    inference_object_x_range: Tuple[float, float] = (-0.25, 0.25)
    inference_object_y_range: Tuple[float, float] = (-0.25, 0.25)
    inference_objects: Optional[dict] = field(default_factory=lambda: {
        "cube": {"pos": "random", "color": "red", "range": [(-0.22, -0.08), (-0.12, 0.12)]},
        "bin": {"pos": (0.06, 0.0), "bin_size": (0.16, 0.20, 0.10)},
        "can": {"pos": (0.13, 0.21), "color": "silver"},
        "bottle": (-0.15, -0.24),
        "milk": (0.15, -0.20),
    })
    inference_camera_size: int = 384
    inference_plan_length_n: int = 4
    inference_max_steps: int = 300
    inference_stereo_use_table_z: bool = False
    inference_stereo_baseline: float = 0.50
    run_pre_post_inference_eval: bool = True
    inference_eval_trials: int = 5
    grpo_group_size: int = 4
    grpo_use_binary_success_reward: bool = True
    planner_reward_success_bonus: float = 1.0
    planner_reward_lift_bonus: float = 0.25
    planner_reward_bin_bonus: float = 0.25
    planner_reward_release_bonus: float = 0.10
    planner_reward_step_penalty: float = 0.001

cfg = RLConfig()
set_seed(cfg.seed)
print(cfg)
if cfg.use_4bit and not torch.cuda.is_available():
    print("Warning: use_4bit=True without CUDA is not practical for this notebook.")

RLConfig(vlm_model_id='Qwen/Qwen3-VL-4B-Instruct', use_4bit=True, gradient_checkpointing=True, lora_r=16, lora_alpha=32, lora_dropout=0.0, lora_target_modules='all-linear', task_prompt="Pick up the red cube and place it into the wooden bin. The wooden bin is an open-top, four-walled container; its walls are about 10 cm (0.10 m) tall, so the bin rim sits roughly 10 cm above the table surface. Follow these steps in order:\n1. Move the gripper directly above the red cube and align it over the cube in the XY plane.\n2. Descend vertically until the gripper contacts the cube.\n3. Close the gripper to grasp the cube firmly.\n4. Lift the grasped cube straight UP by at least 0.15 m (use move_by offset [0,0,0.15] or more). This raises the cube HIGHER than the 10 cm bin wall - the cube MUST clear the wall before you move it sideways over the bin.\n5. Move the gripper over the wooden bin and hover directly above the bin's opening, still higher than the rim.\n6. Descend vertically so the cube drops

---
## 5. Environment Factory and Observation Helpers

In [5]:
PROPRIO_KEYS = (
    "robot0_eef_pos",
    "robot0_eef_quat",
    "robot0_gripper_qpos",
    "robot0_gripper_qvel",
    "object-state",
)

def make_env(cfg: RLConfig, seed: Optional[int] = None):
    cc = load_composite_controller_config(controller="BASIC")
    env = suite.make(
        cfg.env_name,
        robots=cfg.robot,
        controller_configs=cc,
        has_renderer=False,
        has_offscreen_renderer=True,
        renderer="mujoco",
        camera_names=list(cfg.camera_names),
        camera_heights=cfg.camera_size,
        camera_widths=cfg.camera_size,
        camera_depths=False,
        use_camera_obs=True,
        use_object_obs=True,
        reward_shaping=cfg.reward_shaping,
        control_freq=cfg.control_freq,
        horizon=cfg.horizon,
        seed=cfg.seed if seed is None else seed,
    )
    return env

def extract_images(obs: Dict[str, np.ndarray], cfg: RLConfig) -> List[np.ndarray]:
    images = []
    for cam in cfg.camera_names:
        key = f"{cam}_image"
        if key not in obs:
            raise KeyError(f"Missing image key: {key}")
        images.append(np.ascontiguousarray(obs[key][..., :3]))
    return images

def extract_proprio(obs: Dict[str, np.ndarray]) -> np.ndarray:
    parts = []
    for key in PROPRIO_KEYS:
        if key in obs:
            parts.append(np.asarray(obs[key], dtype=np.float32).reshape(-1))
    if not parts:
        raise KeyError(f"None of the proprio keys are present: {PROPRIO_KEYS}")
    return np.concatenate(parts, axis=0).astype(np.float32)

def step_env(env, action: np.ndarray, repeat: int = 1):
    total_reward = 0.0
    obs = None
    done = False
    info = {}
    for _ in range(repeat):
        obs, reward, done, info = env.step(action)
        total_reward += float(reward)
        if done:
            break
    return obs, total_reward, done, info

def inspect_env(cfg: RLConfig):
    env = make_env(cfg)
    try:
        obs = env.reset()
        print(f"Action dim: {env.action_dim}")
        print(f"Observation keys: {sorted(obs.keys())}")
        print(f"Proprio dim: {extract_proprio(obs).shape[0]}")
        for cam, img in zip(cfg.camera_names, extract_images(obs, cfg)):
            print(f"{cam}: {img.shape} {img.dtype}")
    finally:
        env.close()

# Run manually if you want to inspect the env before training.
# inspect_env(cfg)

---
## 6. Qwen LoRA Backbone and Policy Head

In [6]:
class QwenLoRABackbone(nn.Module):
    def __init__(self, cfg: RLConfig):
        super().__init__()
        if cfg.use_4bit and not torch.cuda.is_available():
            raise RuntimeError("4-bit LoRA training requires CUDA in this notebook setup")

        kwargs = {"trust_remote_code": True}
        if torch.cuda.is_available():
            kwargs["device_map"] = "auto"
            if cfg.use_4bit:
                kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                )
                kwargs["torch_dtype"] = torch.bfloat16
            else:
                kwargs["torch_dtype"] = torch.float16
        else:
            kwargs["torch_dtype"] = torch.float32

        base_model = Qwen3VLForConditionalGeneration.from_pretrained(cfg.vlm_model_id, **kwargs)
        if cfg.use_4bit:
            base_model = prepare_model_for_kbit_training(
                base_model,
                use_gradient_checkpointing=cfg.gradient_checkpointing,
            )
        elif cfg.gradient_checkpointing and hasattr(base_model, "gradient_checkpointing_enable"):
            base_model.gradient_checkpointing_enable()

        lora_cfg = LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            target_modules=cfg.lora_target_modules,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        self.model = get_peft_model(base_model, lora_cfg)
        self.model.config.use_cache = False
        self.processor = AutoProcessor.from_pretrained(cfg.vlm_model_id, trust_remote_code=True)

        text_cfg = getattr(self.model.config, "text_config", None)
        self.hidden_size = getattr(text_cfg, "hidden_size", None)
        if self.hidden_size is None:
            self.hidden_size = getattr(self.model.config, "hidden_size", None)
        if self.hidden_size is None:
            raise ValueError("Could not infer Qwen hidden size from config")

        try:
            self.input_device = self.model.get_input_embeddings().weight.device
        except Exception:
            self.input_device = next(self.model.parameters()).device

        if hasattr(self.model, "print_trainable_parameters"):
            self.model.print_trainable_parameters()

    def _build_messages(self, images: Sequence[np.ndarray], task_prompt: str):
        content = []
        for img in images:
            content.append({"type": "image", "image": Image.fromarray(img)})
        content.append(
            {
                "type": "text",
                "text": (
                    f"Task: {task_prompt}\n"
                    "Encode the current scene for low-level robot control."
                ),
            }
        )
        return [{"role": "user", "content": content}]

    def _encode_single(self, images: Sequence[np.ndarray], task_prompt: str) -> torch.Tensor:
        conversation = self._build_messages(images, task_prompt)
        inputs = self.processor.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(self.input_device)
        inputs.pop("token_type_ids", None)
        outputs = self.model(
            **inputs,
            use_cache=False,
            output_hidden_states=True,
            return_dict=True,
        )
        hidden = outputs.hidden_states[-1].float()
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return pooled.to(DEVICE)

    def encode_batch(self, image_batches: Sequence[Sequence[np.ndarray]], task_prompts: Sequence[str]) -> torch.Tensor:
        feats = []
        for images, prompt in zip(image_batches, task_prompts):
            feats.append(self._encode_single(images, prompt))
        return torch.cat(feats, dim=0)


---
## 7. Planner-Centric RL Helper Logic

In [7]:
def normalize_rewards(rewards: torch.Tensor) -> torch.Tensor:
    if rewards.numel() <= 1:
        return rewards - rewards.mean()
    return (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-8)


print("Planner-only RL helpers OK")


Planner-only RL helpers OK


---
## 9. Base-Aligned Inference Stack

In [8]:
import contextlib, json, math, re, shutil, sys
from collections import OrderedDict

import absl.logging
from transformers import Owlv2ForObjectDetection, Owlv2Processor
from robosuite.environments.manipulation.lift import Lift
from robosuite.models.arenas import TableArena
from robosuite.models.objects import BoxObject, MujocoXMLObject
from robosuite.models.tasks import ManipulationTask
from robosuite.utils.mjcf_utils import xml_path_completion
from robosuite.utils.placement_samplers import UniformRandomSampler
import robosuite.models.objects as _robo_objects

BASE_INFERENCE_SEED = None

def set_base_inference_seed(seed: int):
    global BASE_INFERENCE_SEED
    BASE_INFERENCE_SEED = seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

absl.logging.set_verbosity(absl.logging.ERROR)
print("Base-style inference imports OK")


Base-style inference imports OK


In [9]:
@dataclass
class BaseInferenceConfig:
    vlm_model_id: str = "Qwen/Qwen3-VL-4B-Instruct"
    use_4bit: bool = True
    env_name: str = "Lift"
    robot: str = "Panda"
    task_type: str = "pick_up"   # "pick_up" (lift cube & hold) | "pick_and_place" (grasp cube, place into bin, release)
    bbox_mode: str = "auto"   # bbox detection: "auto" (Qwen decides) | "always" | "never"
    # ── Stereo camera ──
    left_camera: str = "left_eye"       # injected into MuJoCo XML
    right_camera: str = "right_eye"     # injected into MuJoCo XML
    stereo_use_table_z: bool = False    # use full stereo 3D (XYZ from triangulation)
    # ── Detection ──
    detection_queries: list = None  # OWLv2 text queries (overridden by bbox_target)
    bbox_target: Optional[str] = None  # OWLv2 query text (None = Qwen regenerates it each re-check round)
    object_x_range: tuple = (-0.20, -0.05)  # cube X range (meters, more negative = closer to robot)
    object_y_range: tuple = (-0.10, 0.10)   # cube Y range (meters)
    objects: dict = None          # {name: (x,y)|"random"|{"pos":..,"color":..,<ctor-kw>..}}; None -> single red cube; first entry = task object
    camera_size: int = 384
    stereo_baseline: float = 0.50       # wider baseline improves depth accuracy (was 0.30)
    # ── Planning ──
    plan_length_n: int = 4
    lift_height: float = 0.15
    n_trials: int = 10
    max_steps: int = 150
    seed: int = 42

    def __post_init__(self):
        self.camera_names = [self.left_camera, self.right_camera]
        if self.detection_queries is None:
            self.detection_queries = [self.bbox_target] if self.bbox_target else ["object"]
        if self.objects is None:
            self.objects = {"cube": {"pos": "random", "color": "red"}}

base_cfg = BaseInferenceConfig()
set_base_inference_seed(base_cfg.seed)
print(f"VLM: {base_cfg.vlm_model_id}")
print(f"Camera: {base_cfg.camera_size}x{base_cfg.camera_size}, stereo baseline={base_cfg.stereo_baseline}m")
print(f"Plan: {base_cfg.plan_length_n} actions, max {base_cfg.max_steps} steps")
print(f"Robot: {base_cfg.robot} in {base_cfg.env_name}")
print(f"Stereo: L={base_cfg.left_camera}, R={base_cfg.right_camera}, baseline={base_cfg.stereo_baseline}m, full_3d={not base_cfg.stereo_use_table_z}")
print(f"Seed: {base_cfg.seed}")
print(f"Detection: OWLv2, queries={base_cfg.detection_queries}")

VLM: Qwen/Qwen3-VL-4B-Instruct
Camera: 384x384, stereo baseline=0.5m
Plan: 4 actions, max 150 steps
Robot: Panda in Lift
Stereo: L=left_eye, R=right_eye, baseline=0.5m, full_3d=True
Seed: 42
Detection: OWLv2, queries=['object']


In [10]:
def sync_base_inference_cfg_from_rl(rl_cfg: RLConfig, task: Optional[str] = None, seed: Optional[int] = None):
    base_cfg.vlm_model_id = rl_cfg.vlm_model_id
    base_cfg.use_4bit = rl_cfg.use_4bit
    base_cfg.robot = rl_cfg.robot
    base_cfg.task_type = getattr(rl_cfg, "inference_task_type", "pick_up")
    base_cfg.bbox_mode = getattr(rl_cfg, "inference_bbox_mode", "auto")
    base_cfg.bbox_target = getattr(rl_cfg, "inference_bbox_target", None)
    base_cfg.object_x_range = getattr(rl_cfg, "inference_object_x_range", (-0.20, -0.05))
    base_cfg.object_y_range = getattr(rl_cfg, "inference_object_y_range", (-0.10, 0.10))
    base_cfg.camera_size = getattr(rl_cfg, "inference_camera_size", 384)
    base_cfg.plan_length_n = getattr(rl_cfg, "inference_plan_length_n", 4)
    base_cfg.max_steps = getattr(rl_cfg, "inference_max_steps", 150)
    base_cfg.stereo_use_table_z = getattr(rl_cfg, "inference_stereo_use_table_z", False)
    base_cfg.stereo_baseline = getattr(rl_cfg, "inference_stereo_baseline", 0.50)
    objects = getattr(rl_cfg, "inference_objects", None)
    base_cfg.objects = objects if objects is not None else {"cube": {"pos": "random", "color": "red"}}
    base_cfg.seed = rl_cfg.seed if seed is None else seed
    base_cfg.camera_names = [base_cfg.left_camera, base_cfg.right_camera]
    base_cfg.detection_queries = [base_cfg.bbox_target] if base_cfg.bbox_target else ["object"]
    set_base_inference_seed(base_cfg.seed)
    return rl_cfg.task_prompt if task is None else task

print("Base inference config sync OK")


Base inference config sync OK


In [11]:
DSL_ACTION_TYPES = {"move_to", "move_by", "descend", "rotate", "grasp", "release", "wait"}

def validate_action(action: dict) -> bool:
    if not isinstance(action, dict): return False
    atype = action.get("type")
    if atype not in DSL_ACTION_TYPES: return False

    if atype == "move_by":
        offset = action.get("offset")
        if not isinstance(offset, list) or len(offset) != 3:
            return False
        if not all(isinstance(v, (int, float)) for v in offset):
            return False
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    elif atype == "move_to":
        target = action.get("target")
        if target not in ("object", "above_object"):
            return False
        if "height" in action:
            h = action["height"]
            if not isinstance(h, (int, float)) or h < 0 or h > 1.0: return False
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    elif atype == "descend":
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    elif atype == "rotate":
        rot = action.get("rotation")
        if not isinstance(rot, list) or len(rot) != 3:
            return False
        if any(not isinstance(v, (int, float)) or v < -1 or v > 1 for v in rot):
            return False
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    # grasp, release, wait — no required parameters
    return True

def validate_plan(plan: dict) -> bool:
    if not isinstance(plan, dict): return False
    if "actions" not in plan or not isinstance(plan["actions"], list): return False
    if len(plan["actions"]) == 0: return False
    if not all(validate_action(a) for a in plan["actions"]): return False
    return True

def validate_bbox(bbox) -> bool:
    """Check bbox is exactly 4 normalized values [x1,y1,x2,y2], or valid string tag."""
    if bbox is None: return True
    if isinstance(bbox, str): return bbox in ("no object found", "not necessary")
    if not isinstance(bbox, list): return False
    if len(bbox) != 4: return False
    if any(not isinstance(v, (int, float)) for v in bbox): return False
    if any(v < 0 or v > 1 for v in bbox): return False
    if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]: return False
    return True

def validate_stereo_bbox(plan: dict) -> bool:
    """Validate that a detection result has both left_bbox and right_bbox."""
    if not isinstance(plan, dict): return False
    lb = plan.get("left_bbox")
    rb = plan.get("right_bbox")
    return validate_bbox(lb) and validate_bbox(rb)

# Smoke tests
assert validate_action({"type":"move_to","target":"object"})
assert validate_action({"type":"move_to","target":"above_object","height":0.05,"speed":0.5})
assert validate_action({"type":"move_by","offset":[0,0,0.15],"speed":0.5})
assert not validate_action({"type":"move","offset":[0,0,-0.1],"speed":0.3})  # "move" is REJECTED — must be "move_by"
assert validate_action({"type":"descend"})
assert not validate_action({"type":"move_to","target":"invalid"})
assert not validate_action({"type":"move_by","offset":[0,0]})

_sample = {"reasoning":"test","actions":[
    {"type":"move_to","target":"above_object","height":0.05},
    {"type":"descend"},
    {"type":"grasp"},
    {"type":"move_by","offset":[0,0,0.15],"speed":0.5}]}
assert validate_plan(_sample), "DSL validation failed"
assert validate_stereo_bbox({"left_bbox":[0.3,0.3,0.5,0.5],"right_bbox":[0.4,0.3,0.6,0.5]})
print("DSL definition OK")

DSL definition OK


In [12]:
class VLMPlanner:
    def __init__(
        self,
        model_id: str,
        use_4bit: bool = True,
        model: Optional[torch.nn.Module] = None,
        processor: Optional[AutoProcessor] = None,
    ):
        print(f"Loading Qwen3-VL: {model_id} ...")
        if model is not None:
            if processor is None:
                raise ValueError("processor must be provided when reusing an existing model")
            self.model = model.eval()
            self.processor = processor
            self._call_count = 0
            self._trace_enabled = False
            self._trace_logprobs = []
            print("  VLM loaded (existing backbone)")
            return

        kwargs = {"torch_dtype": torch.float16, "device_map": "auto",
                  "trust_remote_code": True}
        if use_4bit and torch.cuda.is_available():
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
            kwargs["torch_dtype"] = torch.bfloat16
        self.model = Qwen3VLForConditionalGeneration.from_pretrained(
            model_id, **kwargs).eval()
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self._call_count = 0
        self._trace_enabled = False
        self._trace_logprobs = []
        print("  VLM loaded")

    def begin_trace(self):
        self._trace_enabled = True
        self._trace_logprobs = []

    def end_trace(self):
        logs = self._trace_logprobs
        self._trace_enabled = False
        self._trace_logprobs = []
        if not logs:
            return None
        return torch.stack(logs).sum()

    def _score_response_logprob(self, prompt_inputs, response_text: str) -> torch.Tensor:
        response_ids = self.processor.tokenizer(
            response_text,
            return_tensors="pt",
            add_special_tokens=False,
        ).input_ids.to(prompt_inputs["input_ids"].device)
        if response_ids.numel() == 0:
            return torch.zeros((), device=prompt_inputs["input_ids"].device)

        prompt_ids = prompt_inputs["input_ids"]
        prompt_mask = prompt_inputs["attention_mask"]
        full_input_ids = torch.cat([prompt_ids, response_ids], dim=1)
        full_attention_mask = torch.cat(
            [prompt_mask, torch.ones_like(response_ids, dtype=prompt_mask.dtype)],
            dim=1,
        )

        model_inputs = {
            "input_ids": full_input_ids,
            "attention_mask": full_attention_mask,
        }
        for key, value in prompt_inputs.items():
            if key not in ("input_ids", "attention_mask", "token_type_ids"):
                model_inputs[key] = value

        outputs = self.model(
            **model_inputs,
            use_cache=False,
            return_dict=True,
        )
        logits = outputs.logits[:, prompt_ids.shape[1] - 1:-1, :]
        log_probs = F.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)
        return token_log_probs.sum()

    # ─── Core call ──────────────────────────────────────────

    def _call(self, images: dict, prompt: str, max_new_tokens: int = 1024) -> str:
        content = []
        # Current stereo views
        for name in base_cfg.camera_names:
            if name in images:
                content.append({"type": "image", "image": images[name]})
        # Execution history snapshots (oldest first)
        history_keys = sorted([k for k in images.keys() if k.startswith("history_")])
        for key in history_keys:
            content.append({"type": "image", "image": images[key]})
        content.append({"type": "text", "text": prompt})
        conversation = [{"role": "user", "content": content}]
        inputs = self.processor.apply_chat_template(
            conversation, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
        ).to(self.model.device)
        inputs.pop("token_type_ids", None)
        with torch.no_grad():
            if BASE_INFERENCE_SEED is not None:
                torch.manual_seed(BASE_INFERENCE_SEED + self._call_count)
            self._call_count += 1
            ids = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.3, top_p=0.9,
            )
        ids_trim = ids[0][inputs["input_ids"].shape[1]:]
        result = self.processor.decode(ids_trim, skip_special_tokens=True)
        if self._trace_enabled:
            self._trace_logprobs.append(self._score_response_logprob(inputs, result))
        print(f"--- VLM Call #{self._call_count} ---", flush=True)
        if vlm_log_path:
            with open(vlm_log_path, 'a') as f:
                f.write(f"--- VLM Call #{self._call_count} ---\n{result}\n\n")
        return result

    # ─── Phase 1: Stereo bbox detection ─────────────────────

    
    
    def _make_initial_prompt(self, task, steps_done, completed_actions, ee_pos, n_actions,
                            object_3d, gripper_state="open", needs_object=True):
        obj_str = ("unknown (no target detection)" if object_3d is None
                   else f"({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})")
        padded = list(completed_actions[-5:]) if completed_actions else []
        while len(padded) < 5: padded.append("---")
        completed_str = "; ".join(padded)
        placeholder_note = " (--- = no action yet)" if "---" in completed_str else ""
        cam_desc = build_images_desc(base_cfg.camera_names)
        return (
            f'[ROLE] Follow the rules below, use the specified action primitives, and output your plan in the required JSON format.\n'
            f'Your task: "{task}"\n'
            'Robot: Panda 7-DOF + parallel gripper. Controller: OSC_POSE (EE position control).\n'
            '\n'
            f'[STATUS] Step {steps_done}/{base_cfg.max_steps}. Task "{task}" is still NOT complete.\n'
            f'{len(completed_actions)} actions executed — still NOT complete.\n'
            + '[CURRENT STATE] (coordinates are approximate — cross-check with images)\n'
            f'  EE position: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})\n'
            f'  Target 3D: {obj_str}  (move_to "target" is the keyword "object"/"above_object", NOT these coordinates)\n'
            f'  Gripper: {gripper_state}\n'
            f'  Recent actions: {completed_str}{placeholder_note}\n'
            '\n' + cam_desc + '\n'
            + _COORDS_DESC + '\n' + _ACTION_REFERENCE + '\n'
            + _build_availability_note(object_3d, needs_object) + '\n'
            + _STRATEGY + '\n'
            + f'[OUTPUT] You MUST output EXACTLY {n_actions} actions — fill every slot. Use wait for any slot whose\n'
            f'  real action you cannot confirm yet; it will be handled in the next re-check round.\n'
            + _SHARED_RULES + '\n'
            + _build_example(object_3d, n_actions))

    def _make_closedloop_prompt(self, task, steps_done, completed_actions, ee_pos, n_actions,
                                object_3d, gripper_state="unknown", history_desc=None, needs_object=True):
        obj_str = ("unknown (no target detection)" if object_3d is None
                   else f"({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})")
        padded = list(completed_actions[-5:]) if completed_actions else []
        while len(padded) < 5: padded.append("---")
        completed_str = "; ".join(padded)
        placeholder_note = " (--- = no action yet)" if "---" in completed_str else ""
        cam_desc = build_images_desc(base_cfg.camera_names)
        hist_block = ("\n" + history_desc) if history_desc else ""
        return (
            f'[ROLE] Follow the rules below, use the specified action primitives, and output your plan in the required JSON format.\n'
            'A previous sequence of actions was executed but the task is still not complete. You must continue executing.\n'
            f'Your task: "{task}"\n'
            'Robot: Panda 7-DOF + parallel gripper. Controller: OSC_POSE (EE position control).\n'
            '\n'
            f'[STATUS] Step {steps_done}/{base_cfg.max_steps}. Task "{task}" is still NOT complete.\n'
            f'{len(completed_actions)} actions executed — still NOT complete.\n'
            + '[CURRENT STATE] (coordinates are approximate — cross-check with images)\n'
            f'  EE position: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})\n'
            f'  Target 3D: {obj_str}  (move_to "target" is the keyword "object"/"above_object", NOT these coordinates)\n'
            f'  Gripper: {gripper_state}\n'
            f'  Recent actions: {completed_str}{placeholder_note}\n'
            '\n' + cam_desc + hist_block + '\n'
            + _COORDS_DESC + '\n' + _ACTION_REFERENCE + '\n'
            + _build_availability_note(object_3d, needs_object) + '\n'
            + _STRATEGY + '\n'
            + f'[OUTPUT] You MUST output EXACTLY {n_actions} actions — fill every slot. Use wait for any slot whose\n'
            f'  real action you cannot confirm yet; it will be handled in the next re-check round.\n'
            + _SHARED_RULES + '\n'
            + _build_example(object_3d, n_actions))

    def analyze_task(self, left_img, right_img, task: str) -> tuple:
        """First VLM call: decide whether this task needs object detection, and name the target.
        Returns (needs_object_detection: bool, target_description: str | None).
        On any parse failure defaults to (True, None) so manipulation tasks never skip detection."""
        images = {base_cfg.left_camera: Image.fromarray(left_img),
                  base_cfg.right_camera: Image.fromarray(right_img)}
        prompt = (
            '[TASK ANALYSIS]\n'
            f'User task: "{task}"\n'
            'Look at the stereo images and the task. Decide whether completing this task requires '
            'detecting and locating a SPECIFIC physical object in the scene.\n'
            'YES (needs_object_detection=true): the task asks to pick up, grasp, push, touch, point at, '
            'or otherwise manipulate a named object (e.g. a cube, block, or cup).\n'
            'NO  (needs_object_detection=false): the task only needs the arm to move or rotate freely with '
            'NO specific object target (e.g. dance, wave, move around, rotate in place).\n'
            'Output JSON ONLY:\n'
            '{"reasoning":"...","needs_object_detection":true,"target_description":"red cube"}\n'
            '- target_description: the SINGLE object name to locate first when detection is needed (e.g. "red cube"). '
            'If the task involves several objects, name only the one needed FIRST — later targets are located in '
            'subsequent re-check rounds, one per round. Set to null when needs_object_detection is false.'
        )
        raw = self._call(images, prompt, max_new_tokens=256)
        parsed = self._extract_json(raw)
        if isinstance(parsed, dict) and "needs_object_detection" in parsed:
            needs = bool(parsed.get("needs_object_detection"))
            target = parsed.get("target_description")
        else:
            print("  [TASK ANALYSIS] parse failed, defaulting to needs=true", flush=True)
            needs, target = True, None
        if isinstance(target, str):
            target = target.strip()
            if not target or len(target) > 50:
                target = None
        else:
            target = None
        print(f"  [TASK ANALYSIS] needs_detection={needs}, target={target!r}", flush=True)
        if vlm_log_path:
            with open(vlm_log_path, 'a') as f:
                f.write(f"  [TASK ANALYSIS] needs={needs}, target={target!r}\n\n")
        return needs, target

    def regenerate_query(self, left_img, right_img, task, steps_done, completed_actions, ee_pos,
                         gripper_state="unknown", history_images=None, history_desc=None):
        """Per re-check round: given current execution state, output EXACTLY ONE OWLv2 query naming the
        single object to locate next. Multi-target tasks are handled one object per round (never a list).
        Returns the query string, or None on parse failure (caller keeps the previous round's query)."""
        images = {base_cfg.left_camera: Image.fromarray(left_img),
                  base_cfg.right_camera: Image.fromarray(right_img)}
        if history_images:
            for k, v in history_images.items():
                if isinstance(v, np.ndarray):
                    history_images[k] = Image.fromarray(v)
            images.update(history_images)
        padded = list(completed_actions[-5:]) if completed_actions else []
        while len(padded) < 5: padded.append("---")
        completed_str = "; ".join(padded)
        placeholder_note = " (--- = no action yet)" if "---" in completed_str else ""
        cam_desc = build_images_desc(base_cfg.camera_names)
        hist_block = ("\n" + history_desc) if history_desc else ""
        prompt = (
            '[QUERY RE-CHECK]\n'
            f'User task: "{task}"\n'
            'You are mid-execution in a closed loop that re-checks every few actions and re-plans. The task\n'
            'may take several re-check rounds to finish, and locating different objects is done ONE object\n'
            'per round — never more.\n'
            f'[STATUS] Step {steps_done}/{base_cfg.max_steps}. {len(completed_actions)} actions executed so far.\n'
            f'  EE position: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})\n'
            f'  Gripper: {gripper_state}\n'
            f'  Recent actions: {completed_str}{placeholder_note}\n'
            '\n' + cam_desc + hist_block + '\n'
            '[DECIDE] Using the images (current views + history snapshots) and the completed work above,\n'
            'name the SINGLE physical object whose location is still needed to keep progressing toward the task.\n'
            'If the object you have been tracking is already secured (e.g. it has been grasped and is held off\n'
            'the table) or its location is already known and stable, SWITCH to the next object the task requires.\n'
            'Do not keep re-locating an object you already have. One object only — later rounds handle the rest.\n'
            'Output JSON ONLY:\n'
            '{"reasoning":"why this object is the right next target, given what is already done","query":"<the next object>"}\n'
            '- "query": ONE short noun phrase naming exactly ONE object. Name the actual next object for THIS\n'
            '  task from the images/task text — do NOT copy the placeholder or any example. Never a list, never\n'
            '  more than one object.'
        )
        raw = self._call(images, prompt, max_new_tokens=160)
        parsed = self._extract_json(raw)
        query = None
        if isinstance(parsed, dict):
            query = parsed.get("query") or parsed.get("target_description")
        if isinstance(query, str):
            query = query.strip().strip('"').strip("'")
            if not query or len(query) > 50:
                query = None
        else:
            query = None
        print(f"  [QUERY RE-CHECK] next target={query!r}", flush=True)
        if vlm_log_path:
            with open(vlm_log_path, 'a') as f:
                f.write(f"  [QUERY RE-CHECK] next target={query!r}\n\n")
        return query

    def plan_initial(self, left_img, right_img, task, steps_done, completed_actions, ee_pos,
                     n_actions=4, object_3d=None, gripper_state="open", needs_object=True):
        images = {base_cfg.left_camera: Image.fromarray(left_img),
                  base_cfg.right_camera: Image.fromarray(right_img)}
        prompt = self._make_initial_prompt(task, steps_done, completed_actions, ee_pos,
                                           n_actions, object_3d, gripper_state=gripper_state,
                                           needs_object=needs_object)
        return self._plan_with_retry(images, prompt, n_actions, indent="  ", closedloop=False)

    def plan_closedloop(self, left_img, right_img, task, steps_done, completed_actions, ee_pos,
                        n_actions, object_3d, gripper_state="unknown", history_images=None, history_desc=None,
                        needs_object=True):
        images = {base_cfg.left_camera: Image.fromarray(left_img),
                  base_cfg.right_camera: Image.fromarray(right_img)}
        # Convert any raw numpy history frames to PIL for consistent type
        if history_images:
            for k, v in history_images.items():
                if isinstance(v, np.ndarray):
                    history_images[k] = Image.fromarray(v)
            images.update(history_images)
        prompt = self._make_closedloop_prompt(
            task, steps_done, completed_actions, ee_pos, n_actions,
            object_3d, gripper_state=gripper_state, history_desc=history_desc,
            needs_object=needs_object)
        return self._plan_with_retry(images, prompt, n_actions, indent="    ", closedloop=True)

    @staticmethod
    def _action_hint(a):
        """Targeted guidance for the most common action-validation failures, shown back on retry."""
        if not isinstance(a, dict) or a.get("type") not in DSL_ACTION_TYPES:
            return f'Unknown/missing "type". Use one of: {", ".join(DSL_ACTION_TYPES)}.'
        t = a["type"]
        if t == "move_to" and a.get("target") not in ("object", "above_object"):
            return ('move_to "target" must be exactly "object" or "above_object" (a KEYWORD) — '
                    'NEVER coordinates or a tuple; the system computes exact XYZ from the detected object.')
        if t == "move_by" and not (isinstance(a.get("offset"), list) and len(a["offset"]) == 3):
            return 'move_by needs "offset":[dx,dy,dz] as three numbers (meters).'
        if t == "rotate" and not (isinstance(a.get("rotation"), list) and len(a["rotation"]) == 3):
            return 'rotate needs "rotation":[roll,pitch,yaw], each in [-1,1].'
        return ""

    def _plan_with_retry(self, images, prompt, n_actions, indent="  ", closedloop=False):
        """Call the VLM and retry on malformed/short plans (shared by plan_initial & plan_closedloop).
        Returns a validated plan with exactly n_actions actions, or None after 10 attempts.
        indent: console-print depth (initial=2sp, closedloop=4sp); closedloop enables the stronger
        retry message used when a response carries an 'error' field."""
        attempt = 0
        while True:
            if attempt >= 10:
                tag = "closed-loop" if closedloop else "initial"
                print(f"{indent}[PLAN] gave up after 10 attempts ({tag})", flush=True)
                return None
            raw = self._call(images, prompt, max_new_tokens=768)
            plan = self._extract_json(raw)
            if plan is not None and validate_plan(plan):
                if len(plan["actions"]) != n_actions:
                    err = f"expected {n_actions} actions, got {len(plan['actions'])}"
                    print(f"{indent}[PLAN] retry {attempt+1}: {err}", flush=True)
                    prompt += f'\n\nERROR: {err}. You MUST output exactly {n_actions} actions.'
                    attempt += 1
                    continue
                rsn = plan.get("reasoning", "")
                if rsn: print(f"{indent}[PLAN] {rsn}", flush=True)
                print(f"{indent}[PLAN] actions ({attempt+1}):", flush=True)
                for a in plan["actions"]: print(f"{indent}  {json.dumps(a)}", flush=True)
                return plan
            err = "unknown"
            hint = ""
            if plan is None: err = "json_parse_failed"
            elif not isinstance(plan.get("actions"), list): err = "no_actions"
            elif len(plan["actions"]) == 0 and "error" in plan:
                err = f"error_field: {plan['error'][:120]}"
            else:
                bad_idx = next((i for i, a in enumerate(plan["actions"]) if not validate_action(a)), None)
                if bad_idx is not None:
                    bad = plan["actions"][bad_idx]
                    err = f"invalid_action_{bad_idx}: {json.dumps(bad)}"
                    hint = VLMPlanner._action_hint(bad)
                else:
                    err = "empty_actions"
            print(f"{indent}[PLAN] retry {attempt+1}: {err}" + (f" | hint: {hint}" if hint else ""), flush=True)
            if closedloop and "error" in err:
                prompt += ('\n\nERROR: Your response contains an "error" field and empty actions. '
                           f'You MUST output EXACTLY {n_actions} valid actions. '
                           'NO error field. NO empty array.')
            else:
                prompt += (f"\n\nERROR: {err}. " +
                           (hint + " " if hint else "") +
                           f"Return valid JSON with exactly {n_actions} actions.")
            attempt += 1

    # ─── JSON extraction ────────────────────────────────────

    @staticmethod
    def _find_matching_brackets(text, open_ch, end_ch):
        """Find outermost matched brackets, respecting string literals."""
        start = text.find(open_ch)
        if start < 0:
            return None
        depth = 0
        in_string = False
        escape_next = False
        for i in range(start, len(text)):
            c = text[i]
            if escape_next:
                escape_next = False
                continue
            if c == '\\':
                escape_next = True
                continue
            if c == '"':
                in_string = not in_string
                continue
            if not in_string:
                if c == open_ch:
                    depth += 1
                elif c == end_ch:
                    depth -= 1
                    if depth == 0:
                        return text[start:i + 1]
        return None

    @staticmethod
    def _extract_json(text: str) -> Optional[dict]:
        text = text.strip()

        if text.startswith("{"):
            try:
                result = json.loads(text)
            except json.JSONDecodeError:
                result = None
            if result is not None:
                if isinstance(result.get("actions"), list) and len(result["actions"]) == 0:
                    _pat = re.compile(r'"actions"\s*:\s*(\[)', re.DOTALL)
                    for _m in _pat.finditer(text):
                        _start = _m.start(1)
                        _depth, _i = 1, _start + 1
                        while _depth > 0 and _i < len(text):
                            if text[_i] == '[': _depth += 1
                            elif text[_i] == ']': _depth -= 1
                            _i += 1
                        try:
                            _alt = json.loads(text[_start:_i])
                            if isinstance(_alt, list) and len(_alt) > 0:
                                result["actions"] = _alt
                                break
                        except (json.JSONDecodeError, TypeError):
                            continue
                return result

        if text.startswith("["):
            try:
                arr = json.loads(text)
                if isinstance(arr, list) and len(arr) > 0:
                    if all(isinstance(item, dict) and "type" in item for item in arr):
                        return {"actions": arr}
            except (json.JSONDecodeError, TypeError):
                pass

        obj_str = VLMPlanner._find_matching_brackets(text, '{', '}')
        if obj_str:
            try:
                return json.loads(obj_str)
            except json.JSONDecodeError:
                pass

        arr_str = VLMPlanner._find_matching_brackets(text, '[', ']')
        if arr_str:
            try:
                arr = json.loads(arr_str)
                if isinstance(arr, list) and len(arr) > 0:
                    if all(isinstance(item, dict) and "type" in item for item in arr):
                        return {"actions": arr}
            except (json.JSONDecodeError, TypeError):
                pass

        return None

    def unload(self):
        del self.model
        torch.cuda.empty_cache()
        print("  VLM unloaded")

In [13]:
# ═══════════════════════════════════════════════════════════
# Shared prompt constants (identical in initial + closed-loop prompts)
# ═══════════════════════════════════════════════════════════

_half_b = base_cfg.stereo_baseline / 2.0

_CAMERA_INFO = {
    "left_eye": (
        f'LEFT stereo eye:\n'
        f'   Front-side view from the robot\'s LEFT side (+{_half_b:.2f}m in Y).\n'
        f'   Image is in MuJoCo native orientation: table at BOTTOM, robot arm at TOP.\n'
        f'   LEFT = robot\'s LEFT (+Y), RIGHT = robot\'s RIGHT (−Y).\n'
        f'   Objects appear shifted RIGHT compared to the right eye view.\n'
    ),
    "right_eye": (
        f'RIGHT stereo eye:\n'
        f'   Front-side view from the robot\'s RIGHT side (−{_half_b:.2f}m in Y).\n'
        f'   Same orientation as left eye, offset horizontally.\n'
        f'   Objects appear shifted LEFT compared to the left eye view.\n'
    ),
}


def build_camera_rule(cameras):
    n = len(cameras)
    names = ", ".join(cameras)
    if n == 1:
        return f'CRITICAL: Analyze the camera view ({names}) carefully. Your reasoning MUST reference it.'
    return f'CRITICAL: Analyze BOTH stereo views — LEFT eye ({cameras[0]}) and RIGHT eye ({cameras[1]}). Your reasoning MUST reference both views.'


def build_images_desc(cameras):
    n = len(cameras)
    suffix = "s" if n > 1 else ""
    desc = f'[IMAGES]\n{n} stereo image{suffix}:\n'
    for i, cam in enumerate(cameras, 1):
        info = _CAMERA_INFO.get(cam, "")
        if info:
            desc += f'{i}. {info}'
    return desc


_COORDS_DESC = (
    '[COORDINATES]\n'
    'X = forward (+ = away from robot), Y = robot-left (+ = left), Z = up (table surface z = 0.80)\n'
    'Image coords (MuJoCo native): y=0 = BOTTOM (table surface), y=1 = TOP (robot arm area)\n'
)

_ACTION_REFERENCE = (
    '[ACTIONS — all are JSON objects with "type" and parameters]\n'
    '\n'
    'move_to {"type":"move_to", "target":"object"}\n'
    '  Move EE directly to the detected object\'s 3D position. Code computes exact coordinates.\n'
    '  CRITICAL: "target" is a KEYWORD ("object" or "above_object") — NEVER a coordinate or tuple.\n'
    'move_to {"type":"move_to", "target":"above_object", "height":0.05, "speed":0.5}\n'
    '  Move EE above the detected object by "height" meters. Default height = 0.05 (5cm).\n'
    '  ONLY use move_to when Target 3D is known (not "unknown").\n'
    '\n'
    'move_by {"type":"move_by", "offset":[dx,dy,dz], "speed":float}\n'
    '  Move EE by [dx,dy,dz] meters relative to current position.\n'
    '  Use for simple relative moves with round numbers (e.g., lift up → offset:[0,0,0.15]).\n'
    '  speed 0.1-1.0.\n'
    '\n'
    'descend {"type":"descend"}\n'
    '  Descend vertically (maintain current XY) until arm is blocked by surface contact.\n'
    '  Use for final vertical approach, e.g. before grasping or contacting a surface.\n'
    '\n'
    'rotate {"type":"rotate", "rotation":[dr,dp,dy], "speed":float}\n'
    '  rotation=[dr,dp,dy] normalized [-1,1], relative to current orientation.\n'
    '  speed 0.1-1.0.\n'
    '\n'
    'grasp {"type":"grasp"} — Close gripper fully. No parameters needed.\n'
    '\n'
    'release {"type":"release"} — Open gripper fully. No parameters needed.\n'
    '\n'
    'wait {"type":"wait"} — Do nothing for a fixed duration. No parameters needed.\n'
)


def _build_availability_note(object_3d, needs_object=True):
    """Generate action availability note based on detection state and task needs."""
    if object_3d is not None:
        return (
            '[ACTION AVAILABILITY]\n'
            'move_to is AVAILABLE - Target 3D position is known from stereo detection.\n'
            'For ANY approach to this position (reach the object, or hover above a container to place into it),\n'
            'use move_to (target "object" or "above_object"). The system drives the arm to the EXACT detected\n'
            'coordinates — you do not estimate distance. Use move_by ONLY for relative adjustments UNRELATED to\n'
            'the detected position (e.g. lift straight up after a grasp). NEVER guess a move_by offset toward the\n'
            'detected target — the estimate is wrong and the arm lands short.\n'
        )
    if needs_object:
        return (
            '[ACTION AVAILABILITY]\n'
            'move_to is NOT available - object detection was needed but no target was found.\n'
            'You MUST use move_by with offset for all position changes.\n'
        )
    return (
        '[ACTION AVAILABILITY]\n'
        'This task does NOT require a specific object - no detection was run.\n'
        'move_to is NOT available. Use move_by (offset), rotate, and wait to move the arm freely through space.\n'
    )


def _build_example(object_3d, n_actions):
    """Generate format example based on whether target is detected."""
    if object_3d is not None:
        acts = [
            '{"type":"move_to","target":"above_object","height":0.05}',
            '{"type":"descend"}',
            '{"type":"grasp"}',
            '{"type":"move_by","offset":[0,0,0.15],"speed":0.5}',
        ]
    else:
        acts = [
            '{"type":"move_by","offset":[0.1,0.0,0.05],"speed":0.5}',
            '{"type":"rotate","rotation":[0.3,0.0,0.2],"speed":0.4}',
            '{"type":"move_by","offset":[0.0,0.1,-0.05],"speed":0.5}',
            '{"type":"wait"}',
        ]
    while len(acts) < n_actions:
        acts.append('{"type":"wait"}')
    acts = acts[:n_actions]
    acts_str = ",\n  ".join(acts)
    return (
        '[EXAMPLE — format only, do not copy literally]\n'
        '{"reasoning":"Plan actions based on current state.","actions":[\n'
        f'  {acts_str}\n'
        ']}'
    )


_SHARED_RULES = (
    '[RULES]\n'
    '1. NEVER output "actions":[] or add an "error" field. Both are REJECTED immediately.\n'
    '2. Speed: 0.2-0.4 for precision (descend/grasp/rotate), 0.5-0.8 for gross movement (move_by).\n'
    '3. Vary action types — avoid consecutive repeats of the same type.\n'
    '4. In your "reasoning", analyze the IMAGES (current views + history snapshots) as your primary source. '
    'Coordinates are approximate and for reference only — trust what you see in the images.\n'
    '5. ' + build_camera_rule(base_cfg.camera_names) + '\n'
    '6. JSON ONLY. No text outside the JSON.\n'
)

_STRATEGY = (
    '[STRATEGY]\n'
    'Plan only what you can confidently execute THIS round. The task may need several re-check rounds to\n'
    'finish — you are NOT expected to complete it in a single plan, and later rounds will continue from\n'
    'where you stop. Break complex tasks into small steps.\n'
    '- If manipulating a specific object (pick up, push, touch): use move_to to approach, descend for the\n'
    '  final vertical approach, then grasp/release.\n'
    '- When Target 3D is KNOWN, ALWAYS use move_to (target "object" to reach it, or "above_object" to hover\n'
    '  above it) for the approach — the arm goes to the EXACT detected coordinates. NEVER use move_by to guess\n'
    '  a direction/distance toward a detected target; your offset estimate is wrong and the arm undershoots.\n'
    '  move_by is only for relative adjustments unrelated to a detected position (e.g. lift straight up).\n'
    '- To PLACE a held object INTO a container (bin, box, bowl): the container position is detected, so use\n'
    '  move_to above_object to move directly above the container, then descend into it, then release. Do NOT\n'
    '  nudge toward the container with move_by.\n'
    '- If the task needs free motion (move/rotate the arm, dance, wave) with no specific object: use\n'
    '  move_by (offset) and rotate. Do NOT approach or grasp an object unless the task requires it.\n'
    '- Plan ONLY the next step toward the object that is currently detected. If a later step needs a\n'
    '  DIFFERENT object that has not been located yet, do NOT guess where it is — fill those slots with\n'
    '  wait; the next re-check round will locate that object and continue.\n'
    '- Match your actions to what the task actually asks for.\n'
)

print("Prompt constants OK")

Prompt constants OK


In [14]:
vlm_log_path = None

_base_vlm_planner = None

def get_base_planner():
    global _base_vlm_planner
    if _base_vlm_planner is None:
        gc.collect()
        torch.cuda.empty_cache()
        _base_vlm_planner = VLMPlanner(base_cfg.vlm_model_id, base_cfg.use_4bit)
    return _base_vlm_planner

print("VLMPlanner class defined")

VLMPlanner class defined


In [15]:
# ── BboxDetector (OWLv2 for open-vocabulary object detection) ──

class BboxDetector:
    """Open-vocabulary object detector using OWLv2.
    
    Replaces Qwen3-VL's bbox detection phase. OWLv2 is natively
    supported in transformers (no trust_remote_code needed) and
    detects objects based on text queries.
    """

    def __init__(self, model_id: str = "google/owlv2-base-patch16-ensemble"):
        print(f"Loading OWLv2: {model_id} ...")
        self.processor = Owlv2Processor.from_pretrained(model_id)
        self.model = Owlv2ForObjectDetection.from_pretrained(model_id).eval()
        if torch.cuda.is_available():
            self.model = self.model.cuda()
        self._img_size = None
        print("  OWLv2 loaded")

    @torch.no_grad()
    def detect(self, left_img, right_img, queries=None, threshold=0.05):
        """Detect target object in both stereo views.
        
        Args:
            left_img: numpy array (H, W, 3) RGB, MuJoCo native orientation
            right_img: numpy array (H, W, 3) RGB, MuJoCo native orientation
            queries: list of text queries (e.g. ["red cube", "cube"])
            threshold: confidence threshold for detections
        
        Returns:
            (left_bbox, right_bbox, reasoning) or None
            Each bbox is [x1, y1, x2, y2] in normalized MuJoCo-native coords [0,1]
            (y=0 = BOTTOM of image = table surface, same as backprojection expects)
        """
        if queries is None:
            queries = base_cfg.detection_queries

        H, W = left_img.shape[:2]
        self._img_size = H

        # OWLv2 receives images as PIL or numpy arrays.
        # The image is in MuJoCo native orientation: row 0 = bottom of scene = table surface.
        # When converted to PIL, row 0 becomes image top, so the table appears at image top.
        # OWLv2 outputs bboxes in PIL image coords (y=0 = top = table surface).
        # This happens to match MuJoCo native (y=0 = table surface) — no flip needed.
        # See the coordinate analysis in git history for full derivation.
        left_pil = Image.fromarray(left_img)
        right_pil = Image.fromarray(right_img)

        # Each image gets the same list of text queries
        texts = [queries, queries]

        inputs = self.processor(
            text=texts,
            images=[left_pil, right_pil],
            return_tensors="pt",
        ).to(self.model.device)

        outputs = self.model(**inputs)

        # Post-process to get pixel-coordinate bboxes
        target_sizes = torch.tensor([[H, W], [H, W]]).to(self.model.device)
        results = self.processor.post_process_grounded_object_detection(
            outputs, target_sizes=target_sizes, threshold=threshold
        )

        left_bbox = self._best_bbox(results[0], W, H)
        right_bbox = self._best_bbox(results[1], W, H)

        if left_bbox is None or right_bbox is None:
            reason_parts = []
            if left_bbox is None: reason_parts.append("left no detection")
            if right_bbox is None: reason_parts.append("right no detection")
            print(f"  [OWLv2] {{', '.join(reason_parts)}} (queries={queries}, threshold={threshold})", flush=True)
            return None

        score_left = results[0]["scores"].max().item()
        score_right = results[1]["scores"].max().item()
        print(f"  [OWLv2] L={left_bbox} (score={score_left:.3f})", flush=True)
        print(f"  [OWLv2] R={right_bbox} (score={score_right:.3f})", flush=True)

        reasoning = f"OWLv2 detection (q={queries}, scores={score_left:.2f}/{score_right:.2f})"
        return left_bbox, right_bbox, reasoning

    @staticmethod
    def _best_bbox(result, img_w, img_h):
        """Extract highest-scoring bbox, convert to normalized MuJoCo coords.
        
        OWLv2 returns pixel-coord bboxes in PIL convention (y=0 = top of image).
        Due to MuJoCo's OpenGL row-0 = bottom rendering, the PIL image has the
        table at the top. So OWLv2's y=0 maps to MuJoCo's y=0 (table surface).
        No flip needed. We just normalize to [0, 1].
        """
        if len(result["scores"]) == 0:
            return None
        best_idx = result["scores"].argmax().item()
        box = result["boxes"][best_idx].cpu().numpy()  # [x1, y1, x2, y2] pixel coords
        score = result["scores"][best_idx].item()

        # Normalize to [0, 1]
        bbox_norm = [
            float(box[0] / img_w),
            float(box[1] / img_h),
            float(box[2] / img_w),
            float(box[3] / img_h),
        ]

        # Validate
        if bbox_norm[2] <= bbox_norm[0] or bbox_norm[3] <= bbox_norm[1]:
            print(f"  [OWLv2] invalid bbox: {bbox_norm} (raw pixel box: {box})", flush=True)
            return None
        return bbox_norm

    def unload(self):
        del self.model
        del self.processor
        torch.cuda.empty_cache()
        print("  OWLv2 unloaded")


_base_detector = None

def get_base_detector():
    """Singleton accessor for the OWLv2 bbox detector."""
    global _base_detector
    if _base_detector is None:
        gc.collect()
        torch.cuda.empty_cache()
        _base_detector = BboxDetector()
    return _base_detector

print("BboxDetector class defined")


BboxDetector class defined


---
## 10. Base Geometry and Controller Utilities

In [16]:
# ── 7a. Rendering & Helper Utilities ──

@contextlib.contextmanager
def _silence_stderr():
    """Redirect stderr at both Python and OS FD level."""
    old_stderr = sys.stderr
    null_file = open(os.devnull, "w")
    sys.stderr = null_file
    null_fd = os.open(os.devnull, os.O_WRONLY)
    old_fd = os.dup(2)
    os.dup2(null_fd, 2)
    os.close(null_fd)
    try:
        yield
    finally:
        os.dup2(old_fd, 2)
        os.close(old_fd)
        sys.stderr = old_stderr
        null_file.close()


def render_rgb(env, camera="agentview"):
    """Render camera image in MuJoCo native orientation.

    Returns a contiguous uint8 array (H×W×3).
    MuJoCo OpenGL convention: origin at bottom-left.
      y=0 in image → bottom of view → near camera → table surface
      y=max in image → top of view → far from camera → robot arm area
    Both VLM and video display use this same raw orientation.
    """
    if camera is None:
        return None
    obs = env._get_observations(force_update=True)
    return np.ascontiguousarray(obs[f"{camera}_image"][..., :3])


print("Helpers OK")

Helpers OK


In [17]:
# ── 7b. Stereo Camera Injection ──

import shutil

# Convergence target: table surface center
_STEREO_TARGET = np.array([0.0, 0.0, 0.80])


def _lookat_quat(cam_pos, target, world_up=None):
    """Compute MuJoCo quaternion (w x y z) for camera at cam_pos looking at target."""
    if world_up is None:
        world_up = np.array([0.0, 0.0, 1.0])
    cam_pos = np.asarray(cam_pos, dtype=float)
    target = np.asarray(target, dtype=float)

    forward = target - cam_pos
    forward /= np.linalg.norm(forward)

    right = np.cross(world_up, forward)
    right /= np.linalg.norm(right)

    up = np.cross(forward, right)

    # MuJoCo cam_R columns: [right, -up, -forward]  (camera looks along -Z)
    # Negate up so det(R)=+1 (right-handed frame).
    R = np.column_stack([right, -up, -forward])

    # Rotation matrix → quaternion (w, x, y, z)
    tr = R[0, 0] + R[1, 1] + R[2, 2]
    if tr > 0:
        s = 0.5 / np.sqrt(tr + 1.0)
        w = 0.25 / s
        x = (R[2, 1] - R[1, 2]) * s
        y = (R[0, 2] - R[2, 0]) * s
        z = (R[1, 0] - R[0, 1]) * s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
        w = (R[2, 1] - R[1, 2]) / s
        x = 0.25 * s
        y = (R[0, 1] + R[1, 0]) / s
        z = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
        w = (R[0, 2] - R[2, 0]) / s
        x = (R[0, 1] + R[1, 0]) / s
        y = 0.25 * s
        z = (R[1, 2] + R[2, 1]) / s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
        w = (R[1, 0] - R[0, 1]) / s
        x = (R[0, 2] + R[2, 0]) / s
        y = (R[1, 2] + R[2, 1]) / s
        z = 0.25 * s

    return f"{w:.6f} {x:.6f} {y:.6f} {z:.6f}"


def _inject_stereo_cameras_xml():
    """Inject left_eye and right_eye cameras into robosuite's table_arena.xml.
    Cameras converge toward the table center. Original file is backed up and restored."""
    xml_path = os.path.join(
        os.path.dirname(suite.__file__),
        'models', 'assets', 'arenas', 'table_arena.xml')
    with open(xml_path) as f:
        original = f.read()

    backup = xml_path + '.bak'
    shutil.copy2(xml_path, backup)

    half_b = base_cfg.stereo_baseline / 2.0
    x, z = 0.5, 1.35
    pos_L = [x, half_b, z]
    pos_R = [x, -half_b, z]
    quat_L = _lookat_quat(pos_L, _STEREO_TARGET)
    quat_R = _lookat_quat(pos_R, _STEREO_TARGET)

    stereo_block = (
        f'\n    <!-- Binocular stereo cameras (auto-injected, converge to table center) -->\n'
        f'    <camera mode="fixed" name="left_eye"  pos="{x} {half_b:.4f} {z}" quat="{quat_L}"/>\n'
        f'    <camera mode="fixed" name="right_eye" pos="{x} {-half_b:.4f} {z}" quat="{quat_R}"/>\n'
    )
    modified = original.replace('</worldbody>', stereo_block + '</worldbody>')
    # Enable shadows in the scene lighting
    modified = modified.replace('castshadow="false"', 'castshadow="true"')
    with open(xml_path, 'w') as f:
        f.write(modified)
    return xml_path, backup


def _restore_arena_xml(xml_path, backup_path):
    """Restore the original arena XML after env creation."""
    if os.path.exists(backup_path):
        shutil.move(backup_path, xml_path)


print("Stereo injection OK")

Stereo injection OK


In [18]:
# ── 7c. Camera Parameters & Backprojection ──

def get_camera_params(env, camera="agentview", size=384):
    """Extract intrinsic matrix K, position, rotation, and table Z for a camera."""
    cam_id = env.sim.model.camera_name2id(camera)
    fov = float(env.sim.model.cam_fovy[cam_id])
    f = size / (2.0 * math.tan(math.radians(fov) / 2.0))
    K = np.array([[f, 0, size / 2], [0, f, size / 2], [0, 0, 1]])
    cam_pos = env.sim.model.cam_pos[cam_id].copy()
    cam_R = env.sim.model.cam_mat0[cam_id].reshape(3, 3).copy()
    table_z = float(env.model.mujoco_arena.table_offset[2])
    return K, cam_pos, cam_R, table_z


def get_stereo_params(env):
    """Get camera params for both stereo eyes plus table Z."""
    K_L, pos_L, R_L, tz = get_camera_params(env, camera=base_cfg.left_camera)
    K_R, pos_R, R_R, _   = get_camera_params(env, camera=base_cfg.right_camera)
    return (K_L, pos_L, R_L), (K_R, pos_R, R_R), tz


def backproject_ray(pixel_u, pixel_v, K, cam_pos, cam_R):
    """Back-project a pixel to a ray in world coordinates.

    Args:
        pixel_u, pixel_v: pixel coords in standard image frame (origin top-left, +u right, +v down)
        K: 3x3 intrinsic matrix
        cam_pos: camera position in world frame
        cam_R: camera-to-world rotation (cam_mat0 reshaped to 3x3)

    Returns:
        (origin, direction) tuple — ray in world coordinates
    """
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    # Image is standard orientation (origin top-left, +u right, +v down).
    # Camera looks along -Z, so the ray through pixel (u,v) in camera frame is [(u-cx)/fx, (v-cy)/fy, -1].
    d = cam_R @ np.array([(pixel_u - cx) / fx, (pixel_v - cy) / fy, -1.0])
    d = d / np.linalg.norm(d)
    return cam_pos.copy(), d

print("Backprojection OK")

Backprojection OK


In [19]:
# ── 7d. Stereo Triangulation ──

def triangulate_rays(origin_L, d_L, origin_R, d_R):
    """Find the closest point between two rays via least-squares.

    Solves: minimize |origin_L + t*d_L - origin_R - s*d_R|^2
    """
    W = origin_R - origin_L
    d1d1 = np.dot(d_L, d_L)
    d1d2 = np.dot(d_L, d_R)
    d2d2 = np.dot(d_R, d_R)
    A = np.array([[d1d1, -d1d2], [d1d2, -d2d2]])
    det = np.linalg.det(A)
    if abs(det) < 1e-10:
        return None  # Degenerate: parallel rays
    b = np.array([np.dot(W, d_L), np.dot(W, d_R)])
    ts = np.linalg.solve(A, b)
    pt_L = origin_L + ts[0] * d_L
    pt_R = origin_R + ts[1] * d_R
    gap = np.linalg.norm(pt_L - pt_R)
    if gap > 0.05:  # 5cm sanity threshold
        return None
    return (pt_L + pt_R) / 2.0


def stereo_bbox_to_3d(left_bbox_norm, right_bbox_norm, img_size, env):
    """Triangulate full 3D position from dual stereo bboxes.

    Both bboxes are in normalized MuJoCo-native coords [x1,y1,x2,y2] as output by VLM.
    No coordinate flipping needed — VLM sees the same raw orientation as MuJoCo.
    Returns complete (X, Y, Z) from ray-ray intersection.
    """
    if (left_bbox_norm is None or right_bbox_norm is None
            or not isinstance(left_bbox_norm, (list, tuple)) or len(left_bbox_norm) != 4
            or not isinstance(right_bbox_norm, (list, tuple)) or len(right_bbox_norm) != 4):
        return None

    (K_L, pos_L, R_L), (K_R, pos_R, R_R), tz = get_stereo_params(env)

    # Use bbox CENTER — both rays should point at the same physical point
    left_cx = ((left_bbox_norm[0] + left_bbox_norm[2]) / 2.0) * img_size
    left_cy = ((left_bbox_norm[1] + left_bbox_norm[3]) / 2.0) * img_size
    right_cx = ((right_bbox_norm[0] + right_bbox_norm[2]) / 2.0) * img_size
    right_cy = ((right_bbox_norm[1] + right_bbox_norm[3]) / 2.0) * img_size

    o_L, d_L = backproject_ray(left_cx, left_cy, K_L, pos_L, R_L)
    o_R, d_R = backproject_ray(right_cx, right_cy, K_R, pos_R, R_R)

    point_3d = triangulate_rays(o_L, d_L, o_R, d_R)
    if point_3d is None:
        return None

    # Optional: snap Z to known table height (legacy mode)
    if base_cfg.stereo_use_table_z:
        point_3d[2] = tz

    return point_3d

print("Triangulation OK")

Triangulation OK


In [20]:
# ── Configurable objects + custom Lift environment ──
# base_cfg.objects = { "<built-in name or .xml/.obj path>": (x,y) | "random" | {"pos":(x,y),"color":...}, ... }
#   - EMPTY dict {} / None -> EMPTY TABLE (no objects at all; parameters are the single source of truth)
#   - the FIRST entry becomes the task object (env.cube / env.cube_body_id)
#   - color is set per-entry via "color" (cube defaults to red); any other dict keys (bin_size, ...) forward to the ctor
# See load_new_3Dmodel_experience.md for the external-mesh import workflow.

from collections import OrderedDict
from robosuite.environments.manipulation.lift import Lift
from robosuite.models.objects import BoxObject, MujocoXMLObject
from robosuite.models.arenas import TableArena
from robosuite.models.tasks import ManipulationTask
from robosuite.utils.placement_samplers import UniformRandomSampler
from robosuite.utils.mjcf_utils import xml_path_completion
import robosuite.models.objects as _robo_objects

# color presets (rgba) - "cube" defaults to "red"; any object sets its color via "color" in its dict
OBJECT_COLORS = {
    "red":    [1.0, 0.0, 0.0, 1.0],
    "blue":   [0.0, 0.0, 1.0, 1.0],
    "green":  [0.0, 0.6, 0.0, 1.0],
    "purple": [0.5, 0.0, 0.5, 1.0],
    "yellow": [1.0, 0.85, 0.0, 1.0],
    "white":  [1.0, 1.0, 1.0, 1.0],
    "black":  [0.05, 0.05, 0.05, 1.0],
    "silver": [0.80, 0.80, 0.83, 1.0],
    "gray":   [0.50, 0.50, 0.53, 1.0],
}

# built-in (non-cube) models: dict key -> robosuite object class (native color).
# Every key below is verified importable + instantiable in this robosuite version.
# "cube" is handled separately in _build_object (its color defaults to red; set via "color").
_BUILTIN_XML = {
    # --- food / packaged (mesh assets; recommended pickable objects) ---
    "can":               "CanObject",
    "lemon":             "LemonObject",
    "bottle":            "BottleObject",
    "milk":              "MilkObject",
    "cereal":            "CerealObject",
    "bread":             "BreadObject",
    # --- primitives (parametric; reliable table placement) ---
    "ball":              "BallObject",
    "cylinder":          "CylinderObject",
    "hollow_cylinder":   "HollowCylinderObject",
    "capsule":           "CapsuleObject",
    "cone":              "ConeObject",
    # --- tools / composite (from other task envs; larger footprint, may place oddly) ---
    "hammer":            "HammerObject",
    "pot_with_handles":  "PotWithHandlesObject",
    "bin":               "Bin",
    "hinged_box":        "HingedBoxObject",
    "ratcheting_wrench": "RatchetingWrenchObject",
    "round_nut":         "RoundNutObject",
    "square_nut":        "SquareNutObject",
    "plate_with_hole":   "PlateWithHoleObject",
    "door":              "DoorObject",
}


def _xml_object_from_obj(obj_path, name):
    """Wrap an external .obj mesh in a MuJoCo XML (mesh + material + required sites)
    and return a MujocoXMLObject. Mesh file is resolved relative to the wrapper XML."""
    obj_file = os.path.basename(obj_path)
    xml_path = obj_path + ".wrap.xml"
    xml = (
        '<mujoco model="custom_obj">\n'
        '  <asset>\n'
        f'    <mesh file="{obj_file}" name="mesh" scale="1.0 1.0 1.0"/>\n'
        f'    <material name="mat" rgba="0.8 0.4 0.2 1" reflectance="0.3"/>\n'
        '  </asset>\n'
        '  <worldbody>\n'
        '    <body name="object">\n'
        '      <geom pos="0 0 0" mesh="mesh" type="mesh" material="mat"\n'
        '            density="500" friction="0.95 0.3 0.1"/>\n'
        '    </body>\n'
        '    <site rgba="0 0 0 0" size="0.005" pos="0 0 -0.01" name="bottom_site"/>\n'
        '    <site rgba="0 0 0 0" size="0.005" pos="0 0 0.05" name="top_site"/>\n'
        '    <site rgba="0 0 0 0" size="0.005" pos="0.03 0 0.025" name="horizontal_radius_site"/>\n'
        '  </worldbody>\n'
        '</mujoco>\n'
    )
    with open(xml_path, "w") as _f:
        _f.write(xml)
    return MujocoXMLObject(xml_path, name=name,
                           joints=[dict(type="free", damping="0.0005")],
                           obj_type="all", duplicate_collision_geoms=True)


def _parse_obj_value(val):
    """Normalize a base_cfg.objects value into (pos, color, kwargs, obj_range).

    Accepted forms:
      (x, y) | "random"                  -> ((x,y) or "random", None, {}, None)
      {"pos": (x,y)|"random", "color": "red",
       "range": [(xmin,xmax),(ymin,ymax)],   # optional per-object random box (overrides cfg ranges)
       <object-kwarg>: value, ...}       -> (pos, color, {ctor kwargs}, range)
    "range" is NOT forwarded to the constructor; it fixes the random spawn box for this one
    object (used only when pos=="random"). Extra dict keys (bin_size, wall_thickness, size,
    ...) ARE forwarded to the object constructor."""
    if isinstance(val, dict):
        pos = val.get("pos")
        color = val.get("color")
        obj_range = val.get("range")
        kwargs = {k: v for k, v in val.items() if k not in ("pos", "color", "range")}
        return pos, color, kwargs, obj_range
    return val, None, {}, None


def _recolor_object(obj, rgba):
    """Override a mesh object's baked material/texture with a solid rgba."""
    wb = getattr(obj, "worldbody", None)
    if wb is None:
        return
    for _g in wb.findall(".//geom"):
        _g.attrib.pop("material", None)
        _g.set("rgba", " ".join(str(c) for c in rgba))


# composite classes whose look defaults to a baked texture; a named "color" must flip
# use_texture=False so the rgba actually shows instead of the texture.
_TEXTURED_COMPOSITE = {"bin", "pot_with_handles", "hammer", "hinged_box"}


def _build_object(spec, name, obj_color=None, obj_kwargs=None):
    """Build a MujocoObject from a spec.
    spec: 'cube' | built-in name (can/lemon/bin/...) | path to an existing .xml or .obj file.
    obj_color: optional color preset name (rgba); 'cube' defaults to red when unset.
    obj_kwargs: extra constructor kwargs (e.g. bin_size, wall_thickness, size) from the value dict."""
    obj_kwargs = dict(obj_kwargs or {})
    if str(spec).strip().lower() == "cube":
        rgba = OBJECT_COLORS.get(obj_color, OBJECT_COLORS["red"]) if obj_color else OBJECT_COLORS["red"]
        return BoxObject(name=name, size_min=[0.020, 0.020, 0.020], size_max=[0.022, 0.022, 0.022],
                         rgba=rgba)
    key = str(spec).strip().lower()
    rgba = OBJECT_COLORS.get(obj_color) if obj_color else None
    if key in _BUILTIN_XML:
        cls = getattr(_robo_objects, _BUILTIN_XML[key])
        ctor_kw = dict(obj_kwargs)
        if rgba is not None:
            ctor_kw.setdefault("rgba", rgba)
            if key in _TEXTURED_COMPOSITE:
                ctor_kw.setdefault("use_texture", False)
        try:
            return cls(name=name, **ctor_kw)
        except TypeError:
            # ctor rejects a kwarg (e.g. use_texture on a non-composite, or an unknown size param)
            # -> build bare, then apply color via geom rgba override if requested.
            obj = cls(name=name)
            if rgba:
                _recolor_object(obj, rgba)
            return obj
    path = str(spec)
    if os.path.exists(path) and path.lower().endswith(".xml"):
        obj = MujocoXMLObject(path, name=name,
                              joints=[dict(type="free", damping="0.0005")],
                              obj_type="all", duplicate_collision_geoms=True)
        if rgba:
            _recolor_object(obj, rgba)
        return obj
    if os.path.exists(path) and path.lower().endswith(".obj"):
        obj = _xml_object_from_obj(path, name)
        if rgba:
            _recolor_object(obj, rgba)
        return obj
    raise ValueError(
        f"unknown object spec {spec!r}: use 'cube', a built-in name {list(_BUILTIN_XML)}, "
        "or an existing .xml/.obj path")


class LiftCustom(Lift):
    """Lift task whose object(s) are configured ENTIRELY by a dict.

    objects_spec: {spec: (x,y) | "random" | {"pos":(x,y),"color":...,<kwarg>...}}; first entry is
                  the task object (self.cube). EMPTY dict -> EMPTY TABLE (self.cube=None).
                  Each value's "color" sets the object color (cube defaults to red); any other keys
                  (bin_size, wall_thickness, size, ...) are forwarded to that object's constructor.

    Empty-table guards: _setup_references / reward / _check_success short-circuit when
    self.cube is None so an object-free scene resets, renders, and steps without crashing."""

    def __init__(self, *args, objects_spec=None, **kwargs):
        # No secret default: empty/None -> empty table. Parameters are the single source of truth.
        self._objects_spec = OrderedDict(objects_spec) if objects_spec else OrderedDict()
        super().__init__(*args, **kwargs)

    def _load_model(self):
        # Skip Lift._load_model (hardcodes the red cube); rebuild the scene from base_cfg.objects.
        super(Lift, self)._load_model()
        xpos = self.robots[0].robot_model.base_xpos_offset["table"](self.table_full_size[0])
        self.robots[0].robot_model.set_base_xpos(xpos)
        mujoco_arena = TableArena(table_full_size=self.table_full_size,
                                  table_friction=self.table_friction,
                                  table_offset=self.table_offset)
        mujoco_arena.set_origin([0, 0, 0])

        self._obj_specs = []      # list of (name, spec, pos, obj_range)
        objs = []
        # Bin tracking: if a "bin" entry is present, record its configured size + wall thickness
        # so the controller can compute the bin interior (world coords) for the place-in-bin
        # success test. Defaults match robosuite Bin.__init__.
        self.bin_obj = None
        self.bin_size = (0.3, 0.3, 0.15)
        self.bin_wall_thickness = 0.01
        for i, (spec, val) in enumerate(self._objects_spec.items()):
            pos, color, obj_kwargs, obj_range = _parse_obj_value(val)
            name = "cube" if i == 0 else f"obj{i}"
            obj = _build_object(spec, name, obj_color=color, obj_kwargs=obj_kwargs)
            self._obj_specs.append((name, spec, pos, obj_range))
            objs.append(obj)
            if str(spec).strip().lower() == "bin":
                self.bin_obj = obj
                if "bin_size" in obj_kwargs:
                    self.bin_size = tuple(obj_kwargs["bin_size"])
                if "wall_thickness" in obj_kwargs:
                    self.bin_wall_thickness = float(obj_kwargs["wall_thickness"])
        self.cube = objs[0] if objs else None   # task object = first dict entry; None if empty
        self._all_objects = objs

        # Initial sample spans the full cfg ranges so N objects can obtain a valid
        # (non-overlapping) placement; the exact per-object position is then set
        # in _reset_internal from base_cfg.objects. Empty objs -> sampler returns {}.
        self.placement_initializer = UniformRandomSampler(
            name="ObjectSampler", mujoco_objects=objs,
            x_range=list(base_cfg.object_x_range), y_range=list(base_cfg.object_y_range),
            rotation=None,
            ensure_object_boundary_in_range=False, ensure_valid_placement=True,
            reference_pos=self.table_offset, z_offset=0.01, rng=self.rng)

        self.model = ManipulationTask(
            mujoco_arena=mujoco_arena,
            mujoco_robots=[r.robot_model for r in self.robots],
            mujoco_objects=objs,
        )

    def _setup_references(self):
        # Skip Lift._setup_references (it dereferences self.cube unconditionally); call
        # RobotEnv's version, then set cube_body_id only when a task object exists.
        super(Lift, self)._setup_references()
        self.cube_body_id = (self.sim.model.body_name2id(self.cube.root_body)
                             if self.cube is not None else None)
        # Bin body id (for the pick_and_place success test); None when the scene has no bin.
        self.bin_body_id = (self.sim.model.body_name2id(self.bin_obj.root_body)
                            if getattr(self, "bin_obj", None) is not None else None)

    def _setup_observables(self):
        # Empty table: skip Lift's cube sensors (cube_pos/cube_quat need a cube body id).
        if self.cube is None:
            return super(Lift, self)._setup_observables()
        return super()._setup_observables()

    def reward(self, action=None):
        # Empty table -> no task -> zero reward (never dereference self.cube).
        if self.cube is None:
            return 0.0
        return super().reward(action)

    def _check_success(self):
        if self.cube is None:
            return False
        return super()._check_success()

    def _obj_local_bottom_z(self, obj):
        """Lowest point of obj geometry in its body frame (z < 0 => below the body origin).
        Cube origins sit at the center; can/bottle/milk origins sit high above the base, so a
        uniform body z would clip tall objects. Reads every geom in the object body subtree
        (box/sphere/cylinder/capsule/ellipsoid/mesh) and returns the lowest extent, giving the
        true base for resting the object on the table regardless of origin convention or height."""
        bid = self.sim.model.body_name2id(obj.root_body)
        lowest = float("inf")
        for gi in range(self.sim.model.ngeom):
            gb = int(self.sim.model.geom_bodyid[gi])
            if int(self.sim.model.body_rootid[gb]) != bid:
                continue
            gtype = int(self.sim.model.geom_type[gi])
            gz = float(self.sim.model.geom_pos[gi][2])
            sz = self.sim.model.geom_size[gi]
            if gtype == 6:                       # box
                cand = gz - sz[2]
            elif gtype == 2:                     # sphere
                cand = gz - sz[0]
            elif gtype in (5, 3, 4):             # cylinder / capsule / ellipsoid
                cand = gz - sz[1]
            elif gtype == 7:                     # mesh -> min vertex z, offset by geom local z
                mid = int(self.sim.model.geom_dataid[gi])
                adr = int(self.sim.model.mesh_vertadr[mid])
                num = int(self.sim.model.mesh_vertnum[mid])
                verts = self.sim.model.mesh_vert[adr:adr + num]
                cand = gz + float(verts[:, 2].min())
            else:
                continue
            lowest = min(lowest, cand)
        return lowest if lowest != float("inf") else 0.0

    def _reset_internal(self):
        super()._reset_internal()      # base reset + initial placement (no-op when no objects)
        # Override placement per spec: fixed (x, y) or "random" within cfg ranges.
        for (name, spec, pos, obj_range), obj in zip(self._obj_specs, self._all_objects):
            if pos == "random":
                # Per-object "range" overrides the global cfg ranges so a single object's
                # random box can be restricted (e.g. keep the cube off the bin footprint).
                if obj_range is not None:
                    xr, yr = obj_range
                    x = self.rng.uniform(xr[0], xr[1])
                    y = self.rng.uniform(yr[0], yr[1])
                else:
                    x = self.rng.uniform(base_cfg.object_x_range[0], base_cfg.object_x_range[1])
                    y = self.rng.uniform(base_cfg.object_y_range[0], base_cfg.object_y_range[1])
            elif isinstance(pos, (tuple, list)) and len(pos) == 2:
                x, y = float(pos[0]), float(pos[1])
            else:
                continue
            # Rest the true BASE of each object on the table. Objects differ in height and in where
            # the body origin sits (cube origin = center; can/bottle/milk origins sit well above the
            # base), so a uniform z clips tall objects into the table and MuJoCo pops them out on
            # reset. _obj_local_bottom_z returns the lowest point of the geometry in body frame;
            # placing body_z at table_surface - local_bottom lands that lowest point on the table.
            obj_pos = np.array([x, y, self.table_offset[2] - self._obj_local_bottom_z(obj)])
            self.sim.data.set_joint_qpos(
                obj.joints[0], np.concatenate([obj_pos, np.array([0.0, 0.0, 0.0, 1.0])]))
        self.sim.forward()

print("LiftCustom + object factory OK")


LiftCustom + object factory OK


In [21]:
# ── 7e. Environment Creation ──

def make_base_inference_env(cfg, seed=None, do_reset=True):
    """Create env with stereo cameras injected into the arena XML.
    The original XML is backed up and restored after model compilation."""
    cameras = list(dict.fromkeys(cfg.camera_names))
    xml_path, backup_path = _inject_stereo_cameras_xml()
    try:
        with _silence_stderr():
            cc = load_composite_controller_config(controller="BASIC")
            with open(os.devnull, "w") as _dn:
                with contextlib.redirect_stdout(_dn):
                    env = suite.make(
                        "LiftCustom", robots=cfg.robot, controller_configs=cc,
                        has_renderer=False, has_offscreen_renderer=True, renderer="mujoco",
                        camera_names=cameras,
                        camera_heights=cfg.camera_size, camera_widths=cfg.camera_size,
                        camera_depths=False,
                        use_camera_obs=True, reward_shaping=True, control_freq=20,
                        seed=seed if seed is not None else cfg.seed,
                        objects_spec=cfg.objects,
                    )
            if do_reset:
                env.reset()   # LiftCustom._reset_internal places each object per cfg.objects
    finally:
        _restore_arena_xml(xml_path, backup_path)
    return env

print("make_base_inference_env OK")

make_base_inference_env OK


---
## 11. Base Closed-Loop Controller

In [22]:
from dataclasses import dataclass
@dataclass
class TrialResult:
    success: bool
    steps: int
    error: Optional[str] = None
    plan: Optional[dict] = None

In [23]:
class VideoCapture:
    def __init__(self, fps=8):
        self.frames = []
        self.fps = fps

    def _draw_bbox(self, img_np, bbox_norm, color, label=""):
        """Draw a normalized bbox rectangle on a numpy image (MuJoCo native coords)."""
        from PIL import Image, ImageDraw, ImageFont
        h, w = img_np.shape[:2]
        x1 = int(bbox_norm[0] * w)
        y1 = int(bbox_norm[1] * h)
        x2 = int(bbox_norm[2] * w)
        y2 = int(bbox_norm[3] * h)
        pil = Image.fromarray(img_np)
        draw = ImageDraw.Draw(pil)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        if label:
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
            except Exception:
                font = ImageFont.load_default()
            draw.text((x1 + 2, max(0, y1 - 14)), label, font=font, fill=color)
        return np.array(pil)

    def snap(self, left_img, right_img, phase="", plan=None, t3d=None, msg="",
             step=0, pause=False, bbox=None):
        import cv2
        from PIL import Image, ImageDraw, ImageFont
        left_img = np.ascontiguousarray(left_img)
        right_img = np.ascontiguousarray(right_img)

        # Draw bbox rectangles if provided: bbox = {"left": [x1,y1,x2,y2], "right": [x1,y1,x2,y2]}
        if bbox and isinstance(bbox, dict):
            if "left" in bbox and isinstance(bbox["left"], list):
                left_img = self._draw_bbox(left_img, bbox["left"], (0, 255, 0), "target")
            if "right" in bbox and isinstance(bbox["right"], list):
                right_img = self._draw_bbox(right_img, bbox["right"], (0, 255, 0), "target")

        bar_h = 20

        def _add_label(img_np, label):
            hh, ww = img_np.shape[:2]
            frame = np.zeros((hh + bar_h, ww, 3), dtype=np.uint8)
            frame[:hh] = img_np
            frame[hh:][:] = (40, 40, 50)
            pil_f = Image.fromarray(frame)
            try:
                font_b = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
            except Exception:
                font_b = ImageFont.load_default()
            ImageDraw.Draw(pil_f).text((6, hh + 2), label, font=font_b, fill=(200, 230, 255))
            return np.array(pil_f)

        # Wrap each eye (image + its label bar) in a thin colored border so the two
        # views are visually separated instead of flush. Left=blue, Right=amber.
        eye_border = 4
        labeled = [
            cv2.copyMakeBorder(_add_label(left_img, "Left Eye"),
                               eye_border, eye_border, eye_border, eye_border,
                               cv2.BORDER_CONSTANT, value=(60, 120, 170)),
            cv2.copyMakeBorder(_add_label(right_img, "Right Eye"),
                               eye_border, eye_border, eye_border, eye_border,
                               cv2.BORDER_CONSTANT, value=(170, 120, 60)),
        ]
        disp = np.concatenate(labeled, axis=1)
        h, w = disp.shape[:2]
        info_h = 260
        composite = np.zeros((h + info_h, w, 3), dtype=np.uint8)
        composite[:h] = disp
        composite[h:][:] = (25, 25, 35)

        pil_img = Image.fromarray(composite)
        draw = ImageDraw.Draw(pil_img)
        try:
            font_m = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
            font_s = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
        except Exception:
            font_m = font_s = ImageFont.load_default()

        def _wrap(text, font, max_w):
            words = text.split()
            lines_o, cur = [], ""
            for wd in words:
                test = cur + " " + wd if cur else wd
                if draw.textbbox((0, 0), test, font=font)[2] <= max_w:
                    cur = test
                else:
                    if cur: lines_o.append(cur)
                    cur = wd
            if cur: lines_o.append(cur)
            return lines_o

        margin, max_text_w = 10, w - 2 * 10
        y = h + 8
        draw.text((margin, y), f"Step {step}  {phase}", font=font_m, fill=(255, 200, 100))
        y += 22
        if plan:
            for rl in _wrap(plan.get("reasoning", ""), font_s, max_text_w):
                draw.text((margin, y), rl, font=font_s, fill=(180, 220, 255)); y += 16
            for aidx, act in enumerate(plan.get("actions", [])):
                for al in _wrap(f"  {aidx+1}. {json.dumps(act, ensure_ascii=False)}", font_s, max_text_w):
                    draw.text((margin, y), al, font=font_s, fill=(150, 200, 150)); y += 16
                    if y > h + info_h - 16: break
                if y > h + info_h - 16:
                    draw.text((margin, y), "  ...", font=font_s, fill=(100, 100, 100)); break
        if t3d is not None:
            draw.text((margin, y), f"3D: ({t3d[0]:.3f}, {t3d[1]:.3f}, {t3d[2]:.3f})",
                      font=font_s, fill=(150, 200, 255)); y += 16
        if msg:
            for ml in _wrap(msg, font_s, max_text_w):
                draw.text((margin, y), ml, font=font_s, fill=(200, 200, 200)); y += 16
        self.frames.append(np.array(pil_img))
        if pause:
            for _ in range(12): self.frames.append(np.array(pil_img))  # 1.5s at 8fps

    def save(self, path="output.mp4"):
        n_frames = len(self.frames)
        print(f"  [SAVE] {n_frames} frames, {self.fps} fps", flush=True)
        if not self.frames:
            return path

        # Primary: imageio + imageio-ffmpeg. The bundled ffmpeg ships libx264, giving
        # clean H264 output and avoiding OpenCV's "Encoder not found" stderr spam that
        # OpenCV's own libavcodec (no libx264) prints for avc1/H264/X264 on Kaggle.
        try:
            import imageio
            writer = imageio.get_writer(path, fps=self.fps, codec="libx264",
                                        quality=9, macro_block_size=2)
            for f in self.frames:
                writer.append_data(f)
            writer.close()
            size = os.path.getsize(path)
            if size > 1024:
                print(f"  [SAVE] imageio/libx264: {size/1024:.0f} KB", flush=True)
                return path
        except Exception as e:
            print(f"  [SAVE] imageio libx264 failed ({e}); trying cv2 mp4v", flush=True)

        # Fallback: cv2 with mp4v (MPEG-4 Part 2). Goes straight to mp4v to avoid the
        # avc1/H264/X264 attempts that spam stderr on Kaggle's OpenCV build.
        import cv2
        h, w = self.frames[0].shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(path, fourcc, self.fps, (w, h))
        ok = out.isOpened()
        if ok:
            for f in self.frames:
                out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        out.release()
        if ok and os.path.exists(path) and os.path.getsize(path) > 1024:
            print(f"  [SAVE] cv2/mp4v: {os.path.getsize(path)/1024:.0f} KB", flush=True)
            return path

        print("  [SAVE] All encoders failed!", flush=True)
        return path


In [24]:
class VLMController:
    def __init__(self, env, planner: VLMPlanner, on_frame=None):
        self.env = env
        self.planner = planner
        eef = env.robots[0].eef_site_id
        self._ee_site = eef[list(eef.keys())[0]] if isinstance(eef, dict) else eef
        self._tz = 0.800      # table-surface z (success floor = _tz+0.05)
        self._grasp_z = 0.826  # descend target: pad-center aligns with cube center
        self.on_frame = on_frame
        self._step = 0
        self._action_count = 0
        self._history_frames = []       # list of ({camera: frame}, desc), oldest first, max 2
        self._gripper_closed = False     # track gripper state across actions
        self._target_cache = {}   # query -> 3D position; persists across re-plan rounds WITHIN a
                                  # trial (reset per trial in run_closedloop) so a failed re-detect
                                  # falls back to THIS query's last-known pose, never another
                                  # object's stale coordinates
        self._object_held = False  # latches True once a grasp + lift is observed (cube clearly off the
                                   # table with the gripper closed); stays True through an intentional
                                   # descend-to-place so the planner is never told the grasp failed
        self._actions = []          # recorded per-env.step actions (for offline replay)
        self._frames = []           # recorded frame-event metadata (for offline replay)

    def _ee_pos(self):
        return self.env.sim.data.site_xpos[self._ee_site].copy()

    def _osc_action(self, dxyz, grip, rotation=None):
        if rotation is None: rotation = [0, 0, 0]
        return np.concatenate([np.clip(dxyz / 0.05, -1, 1), np.clip(np.array(rotation, dtype=float), -1, 1), [grip]])

    def _grip_cmd(self):
        """Sticky gripper command: the gripper stays in its current state (closed or open) across
        every move/rotate/wait. Only `grasp` (-> closed) and `release` (-> open) toggle it, so a
        grasped object is NOT dropped when a later move starts."""
        return 1.0 if self._gripper_closed else -1.0

    def _is_success(self):
        """Task success, branching on base_cfg.task_type:
        - pick_and_place: red cube resting INSIDE the bin (xy within the interior, z between
          the floor and the rim — xyz ALL checked) AND the gripper has released it.
        - pick_up (default): cube lifted above the table AND near the end-effector (held)."""
        if base_cfg.task_type == "pick_and_place":
            return self._is_pick_and_place_success()
        cube_pos = self.env.sim.data.body_xpos[self.env.cube_body_id]
        if cube_pos[2] <= self._tz + 0.05:
            return False
        ee = self._ee_pos()
        return np.linalg.norm(ee - cube_pos) < 0.10

    def _is_pick_and_place_success(self):
        """Cube placed inside the bin and released. All three axes are checked:
          - xy: cube center within the bin INTERIOR (clear of the walls);
          - z : cube center between the bin floor and the rim (resting inside, neither held
                above nor sitting on top of the walls);
          - gripper OPEN (a release action was issued)."""
        env = self.env
        if getattr(env, "bin_body_id", None) is None or env.cube_body_id is None:
            return False
        cube_pos = env.sim.data.body_xpos[env.cube_body_id]
        bin_pos = env.sim.data.body_xpos[env.bin_body_id]
        bin_size = np.asarray(env.bin_size, dtype=float)
        wall = float(env.bin_wall_thickness)
        table_z = float(env.table_offset[2])
        # interior half-extents (clear of the walls) minus a small safety margin
        half_x = (bin_size[0] - 2.0 * wall) / 2.0 - 0.005
        half_y = (bin_size[1] - 2.0 * wall) / 2.0 - 0.005
        inside_xy = (abs(cube_pos[0] - bin_pos[0]) < half_x and
                     abs(cube_pos[1] - bin_pos[1]) < half_y)
        floor_top = table_z + wall            # interior floor surface (world z)
        rim_top = table_z + bin_size[2]       # top of the walls (world z)
        inside_z = (cube_pos[2] > floor_top - 0.02) and (cube_pos[2] < rim_top - 0.01)
        gripper_open = not self._gripper_closed
        return bool(inside_xy and inside_z and gripper_open)

    def _ee_over_bin(self):
        """True when the end-effector xy sits within the bin interior (the same interior test as
        _is_pick_and_place_success, applied to the EE rather than the cube). The release guard uses
        this so a pick_and_place `release` only fires over the bin — a premature planner release
        that would drop the cube on the table is refused and the gripper stays closed."""
        env = self.env
        if getattr(env, "bin_body_id", None) is None:
            return False
        ee = self._ee_pos()
        bin_pos = env.sim.data.body_xpos[env.bin_body_id]
        bin_size = np.asarray(env.bin_size, dtype=float)
        wall = float(env.bin_wall_thickness)
        half_x = (bin_size[0] - 2.0 * wall) / 2.0 - 0.005
        half_y = (bin_size[1] - 2.0 * wall) / 2.0 - 0.005
        return bool(abs(ee[0] - bin_pos[0]) < half_x and abs(ee[1] - bin_pos[1]) < half_y)

    def _target_over_bin(self, target_pos):
        """True when target_pos xy is over the bin FOOTPRINT (outer, including the walls) — used by
        the above_object rim-clamp to decide whether a carry-target must clear the rim. Broader than
        _ee_over_bin (which tests the interior) because the approach has to clear the wall too."""
        env = self.env
        if getattr(env, "bin_body_id", None) is None:
            return False
        bin_pos = env.sim.data.body_xpos[env.bin_body_id]
        bin_size = np.asarray(env.bin_size, dtype=float)
        half_x = bin_size[0] / 2.0
        half_y = bin_size[1] / 2.0
        return bool(abs(target_pos[0] - bin_pos[0]) < half_x
                    and abs(target_pos[1] - bin_pos[1]) < half_y)

    def _rim_top(self):
        """World-z of the top of the bin walls (table_z + bin_size[2]), or None if no bin."""
        if getattr(self.env, "bin_body_id", None) is None:
            return None
        return float(self.env.table_offset[2]) + float(np.asarray(self.env.bin_size)[2])

    def _bin_footprint(self):
        """(cx, cy, hx, hy) of the bin OUTER footprint (includes the walls), or None if no bin."""
        if getattr(self.env, "bin_body_id", None) is None:
            return None
        bin_pos = self.env.sim.data.body_xpos[self.env.bin_body_id]
        bin_size = np.asarray(self.env.bin_size, dtype=float)
        return (float(bin_pos[0]), float(bin_pos[1]), bin_size[0] / 2.0, bin_size[1] / 2.0)

    def _carry_crosses_bin(self, target_pos):
        """True when the gripper is CLOSED (carrying) in pick_and_place AND the EE is currently OUTSIDE
        the bin footprint but the straight-line xy path to target_pos enters it. That is the only case
        where a naive straight-line move would drive the held object THROUGH a bin wall: the
        above_object rim-clamp sets the endpoint above the rim but the diagonal PATH still dips under
        the near wall, and a move_by carry bypasses the clamp entirely. When the EE is already over the
        bin opening (an intended descend-to-place) this returns False so the drop-in is not blocked."""
        if base_cfg.task_type != "pick_and_place":
            return False
        if getattr(self.env, "bin_body_id", None) is None:
            return False
        if not self._gripper_closed:
            return False
        fp = self._bin_footprint()
        cx, cy, hx, hy = fp

        def _inside(x, y):
            return abs(x - cx) < hx and abs(y - cy) < hy

        ee = self._ee_pos()
        if _inside(ee[0], ee[1]):
            return False          # already over the opening: a descend-to-place is intended
        # sample the xy segment ee -> target; entering the footprint anywhere means a wall crossing
        for t in np.linspace(0.0, 1.0, 24):
            x = ee[0] + t * (target_pos[0] - ee[0])
            y = ee[1] + t * (target_pos[1] - ee[1])
            if _inside(x, y):
                return True
        return False

    def _all_views(self):
        """Render both stereo camera views (MuJoCo native orientation)."""
        return (render_rgb(self.env, camera=base_cfg.left_camera),
                render_rgb(self.env, camera=base_cfg.right_camera))

    def _step_env(self, action):
        """Wrap env.step so every action vector is recorded for deterministic replay."""
        a = np.asarray(action, dtype=float)
        self._actions.append(a.copy())
        return self.env.step(a)

    def _emit_frame(self, phase, plan=None, t3d=None, msg="", pause=False, bbox=None):
        """Record a frame event for replay (always) and render it live (only if on_frame set)."""
        self._frames.append({
            "step": self._step, "phase": phase, "plan": plan,
            "t3d": None if t3d is None else list(t3d),
            "msg": msg, "pause": pause,
            "bbox": None if bbox is None else {k: list(v) for k, v in bbox.items()},
        })
        if self.on_frame is not None:
            img_L, img_R = self._all_views()
            self.on_frame(img_L, img_R, phase, plan, t3d, msg, self._step, pause=pause, bbox=bbox)

    def _hold_and_capture(self, n=20):
        """After success, hold the EE still for n steps so the video shows the outcome clearly
        instead of cutting off at the instant of success. The gripper keeps its current state
        (closed for pick_up, open for pick_and_place) so a released cube stays put in the bin."""
        hold = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, self._grip_cmd()])
        cube_id = self.env.cube_body_id
        for _ in range(n):
            if self._step >= base_cfg.max_steps:
                break
            self._step += 1
            self._step_env(hold)
            z = float(self.env.sim.data.body_xpos[cube_id][2])
            self._emit_frame("SUCCESS holding", msg=f"cube held at z={z:.3f}")

    def get_recording(self):
        """Return recorded action stream + frame metadata for offline video replay."""
        return {"actions": np.array(self._actions, dtype=float), "frames": self._frames}

    def _record_history(self, action, result_str):
        frames = {cam: render_rgb(self.env, camera=cam) for cam in base_cfg.camera_names}
        desc = f'{action["type"]}(step={self._step},{result_str})'
        self._history_frames.append((frames, desc))
        if len(self._history_frames) > 2:
            self._history_frames.pop(0)

    def _build_history_desc(self):
        if not self._history_frames: return None
        cams = base_cfg.camera_names
        lines = [
            '\n[HISTORY] Execution snapshots at action completion (after current views, oldest first).',
            f'Each entry contains both stereo views ({" + ".join(cams)}).',
            f'{len(self._history_frames)} history entries:'
        ]
        for i, (_, desc) in enumerate(self._history_frames):
            lines.append(f'  History {i}: {desc}')
        return '\n'.join(lines)

    # ─── Shared move-to-position primitive ─────────────────

    def _move_to_position(self, target_pos, speed=0.5):
        """Move EE to target_pos using proportional control with plateau detection.

        CARRY-PATH SAFETY: when carrying an object in pick_and_place and the straight-line path to
        target_pos would cross the bin footprint, lift straight up to a cruise altitude above the rim
        FIRST, then move over. A direct diagonal from a low grasp height to a high bin target passes
        UNDER the rim at the near wall and clips it; the above_object rim-clamp only sets the endpoint,
        it does not route the path. Descending INTO the bin (EE already over the opening) is left
        untouched so an intentional place-descend is never blocked."""
        target_pos = np.asarray(target_pos, dtype=float)
        if self._carry_crosses_bin(target_pos):
            ee0 = self._ee_pos()
            cruise_z = self._rim_top() + 0.05      # clear the rim + held-object half-height
            lift_tgt = np.array([ee0[0], ee0[1], cruise_z])
            print(f"  [CARRY-LIFT] path crosses bin -> lift to cruise z={cruise_z:.3f} before moving over", flush=True)
            st = self._move_to_position(lift_tgt, speed)   # pure vertical; xy unchanged -> no re-trigger
            if st == "max_steps":
                return st
            target_pos = target_pos.copy()
            target_pos[2] = max(target_pos[2], cruise_z)   # hold cruise across the horizontal phase
        stagnant = 0
        prev_dist = np.linalg.norm(self._ee_pos() - target_pos)
        for _ in range(300):
            self._step += 1
            if self._step >= base_cfg.max_steps:
                return "max_steps"
            ee = self._ee_pos()
            dist = np.linalg.norm(ee - target_pos)
            if dist < 0.005:
                self._step_env(self._osc_action(np.zeros(3), self._grip_cmd())); break
            if abs(prev_dist - dist) < 0.001:
                stagnant += 1
                if stagnant > 15:
                    self._step_env(self._osc_action(np.zeros(3), self._grip_cmd())); break
            else:
                stagnant = 0
            prev_dist = dist
            d = target_pos - ee
            act = self._osc_action(d * speed, self._grip_cmd())
            self._step_env(act)
            if self._is_success():
                self._emit_frame("SUCCESS", msg=f"cube lifted to {self.env.sim.data.body_xpos[self.env.cube_body_id][2]:.3f}")
                return "task_done"
            self._emit_frame(f"Move #{self._action_count}",
                             t3d=target_pos, msg=f"dist={dist:.3f} stag={stagnant}")
        return "plateau" if stagnant > 15 else "reached" if dist < 0.005 else "max_steps"

    # ─── Action execution ───────────────────────────────────

    def _execute_action(self, action: dict, object_3d) -> str:
        atype = action["type"]
        speed = action.get("speed", 0.5)

        if atype == "move_to":
            if object_3d is None:
                print("  [WARN] move_to but no object_3d, skipping", flush=True)
                return "skipped"
            target = action.get("target", "object")
            ee_now = self._ee_pos()
            if target == "object":
                target_pos = object_3d.copy()
            elif target == "above_object":
                height = action.get("height", 0.05)
                target_pos = object_3d.copy()
                target_pos[2] += height
                # When carrying an object toward a position over the bin (pick_and_place), raise the
                # approach target above the bin rim so the held object clears the wall on the way in.
                # The planner's height (default 0.05) is sized for a small object and would place a
                # held cube at/below the rim, clipping the wall; clamp to rim_top + margin then.
                if (base_cfg.task_type == "pick_and_place"
                        and getattr(self.env, "bin_body_id", None) is not None
                        and self._gripper_closed
                        and self._target_over_bin(target_pos)):
                    rim_top = float(self.env.table_offset[2]) + float(np.asarray(self.env.bin_size)[2])
                    target_pos[2] = max(target_pos[2], rim_top + 0.04)
            else:
                print(f"  [WARN] unknown move_to target: {target}", flush=True)
                return "unknown_target"
            print(f"  [DEBUG] move_to {target}: EE=({ee_now[0]:.3f},{ee_now[1]:.3f},{ee_now[2]:.3f}) "
                  f"object_3d=({object_3d[0]:.3f},{object_3d[1]:.3f},{object_3d[2]:.3f}) "
                  f"target=({target_pos[0]:.3f},{target_pos[1]:.3f},{target_pos[2]:.3f}) "
                  f"delta=({target_pos[0]-ee_now[0]:.3f},{target_pos[1]-ee_now[1]:.3f},{target_pos[2]-ee_now[2]:.3f})", flush=True)
            return self._move_to_position(target_pos, speed)

        elif atype == "move_by":
            offset = action.get("offset", [0, 0, 0])
            if not isinstance(offset, list) or len(offset) != 3: offset = [0, 0, 0]
            ee_now = self._ee_pos()
            target_pos = ee_now + np.array(offset, dtype=float)
            print(f"  [DEBUG] move_by offset={offset}: EE=({ee_now[0]:.3f},{ee_now[1]:.3f},{ee_now[2]:.3f}) "
                  f"target=({target_pos[0]:.3f},{target_pos[1]:.3f},{target_pos[2]:.3f})", flush=True)
            return self._move_to_position(target_pos, speed)

        elif atype == "descend":
            speed = action.get("speed", 0.3)
            ee = self._ee_pos()
            target_pos = np.array([ee[0], ee[1], self._grasp_z])
            return self._move_to_position(target_pos, speed)

        elif atype == "rotate":
            rot = action.get("rotation", [0, 0, 0])
            if not isinstance(rot, list) or len(rot) != 3: rot = [0, 0, 0]
            max_rot = max(abs(v) for v in rot)
            steps = max(10, min(200, int(80 * max_rot / max(speed, 0.1))))
            for _ in range(steps):
                self._step += 1
                if self._step >= base_cfg.max_steps:
                    return "max_steps"
                self._step_env(self._osc_action(np.zeros(3), self._grip_cmd(), rot))
                if self._is_success():
                    self._emit_frame("SUCCESS", msg=f"cube lifted to {self.env.sim.data.body_xpos[self.env.cube_body_id][2]:.3f}")
                    return "task_done"
                self._emit_frame(f"Move #{self._action_count}: {atype}", msg=f"rot={rot}")
            return "done"

        elif atype == "grasp":
            for i in range(10):
                self._step += 1
                self._step_env(np.array([0, 0, 0, 0, 0, 0, 1.0]))
                if self._is_success():
                    self._gripper_closed = True
                    return "task_done"
                self._emit_frame(f"Move #{self._action_count}: {atype}", msg="grasping")
            self._gripper_closed = True
            return "done"

        elif atype == "release":
            # Release guard (pick_and_place only): refuse to open the gripper unless the EE is over
            # the bin interior. The planner sometimes emits `release` before reaching the bin;
            # without this guard the cube drops on the table and — because there is a single
            # object_3d — the loop can no longer target the cube to re-grasp it. Task types with no
            # bin are unaffected (the guard is a no-op). The blocked slot is consumed as a
            # closed-gripper hold so the action accounting stays consistent.
            if (base_cfg.task_type == "pick_and_place"
                    and getattr(self.env, "bin_body_id", None) is not None
                    and not self._ee_over_bin()):
                print("  [GUARD] release blocked: EE not over bin, keeping gripper closed", flush=True)
                for _ in range(10):
                    self._step += 1
                    if self._step >= base_cfg.max_steps:
                        break
                    self._step_env(self._osc_action(np.zeros(3), self._grip_cmd()))
                return "blocked"
            for _ in range(10):
                self._step += 1
                self._step_env(np.array([0, 0, 0, 0, 0, 0, -1.0]))
            self._gripper_closed = False
            self._object_held = False    # released: the held latch clears with the gripper
            return "done"

        elif atype == "wait":
            for _ in range(10):
                self._step += 1
                self._step_env(self._osc_action(np.zeros(3), self._grip_cmd()))
            return "done"
        return "unknown"

    # ─── Stereo detection + triangulation ───────────────────

    def _detect_and_project(self, task: str, target_desc: str = None) -> tuple:
        # Called only when the task needs an object (gated in run_closedloop).
        # Query priority: target_desc (from task analysis) > base_cfg.bbox_target > generic.
        if target_desc:
            queries = [target_desc]
            print(f"  [DETECT] using target='{target_desc}'", flush=True)
        elif base_cfg.bbox_target is not None:
            queries = [base_cfg.bbox_target]
            print(f"  [DETECT] using bbox_target='{base_cfg.bbox_target}'", flush=True)
        else:
            queries = ["object"]
            print("  [DETECT] no target specified, using generic 'object'", flush=True)

        # Single attempt: OWLv2 detection is deterministic (@torch.no_grad, argmax, no sampling),
        # so retrying within one round returns byte-identical boxes and can never turn a fail into a
        # success. Recovery from a failed re-detect is the caller's job, via the per-query cache.
        img_L, img_R = self._all_views()
        result = get_base_detector().detect(img_L, img_R, queries=queries)
        if result is None:
            print("  [BBOX] OWLv2 returned no detection", flush=True)
            return None, None
        left_bbox, right_bbox, reasoning = result
        object_3d = stereo_bbox_to_3d(left_bbox, right_bbox, base_cfg.camera_size, self.env)
        if object_3d is not None:
            print(f"  [OWLv2] 3D target=({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})", flush=True)
            self._target_cache[queries[0]] = object_3d.copy()  # remember this query's last pose
            bbox_msg = f"L={left_bbox} R={right_bbox}"
            bbox_data = {"left": left_bbox, "right": right_bbox}
            self._emit_frame("Stereo Bbox Detection", msg=bbox_msg, pause=True, bbox=bbox_data)
            return object_3d, (left_bbox, right_bbox)
        print("  [BBOX] triangulation failed (L/R bboxes do not correspond)", flush=True)
        return None, None

    # ─── Main closed-loop ───────────────────────────────────

    def run_closedloop(self, task: str) -> TrialResult:
        n = base_cfg.plan_length_n
        max_st = base_cfg.max_steps
        self.planner._call_count = 0
        self._history_frames = []
        self._gripper_closed = False
        self._object_held = False   # per-trial: held latch resets so a prior trial's state does
                                    # not leak into the next
        self._target_cache = {}     # per-trial: query -> last-known 3D pose (reset so a random
                                    # cube spawn in trial N+1 never leaks trial N's cube pose)
        self._actions = []          # reset recorded action stream for this trial
        self._frames = []           # reset recorded frame metadata for this trial

        # --- Task analysis: does this task need object detection? (decided ONCE, up front) ---
        img_L, img_R = self._all_views()
        if base_cfg.bbox_mode == "never":
            self._needs_object, self._target_desc = False, None
        elif base_cfg.bbox_mode == "always":
            self._needs_object, self._target_desc = True, base_cfg.bbox_target
        else:  # "auto" - Qwen decides whether the task needs an object
            self._needs_object, self._target_desc = self.planner.analyze_task(img_L, img_R, task)
            # An explicit base_cfg.bbox_target overrides the query string whenever detection runs,
            # so pick-up tasks keep their exact OWLv2 query.
            if self._needs_object and base_cfg.bbox_target is not None:
                self._target_desc = base_cfg.bbox_target

        # --- Detection (only when the task needs an object) ---
        if self._needs_object:
            object_3d, bboxes = self._detect_and_project(task, self._target_desc)
            if object_3d is None:
                return TrialResult(False, 0, "no_bbox_detected")
        else:
            object_3d = None
            print("  [DETECT] skipped - task analysis decided no object detection needed", flush=True)

        # Initial plan
        completed_actions = []
        img_L, img_R = self._all_views()
        plan = self.planner.plan_initial(img_L, img_R, task, self._step, completed_actions,
                                         self._ee_pos(), n_actions=n, object_3d=object_3d,
                                         gripper_state="open", needs_object=self._needs_object)
        if plan is None:
            return TrialResult(False, 0, "initial_plan_failed")
        obj_msg = f"object at ({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})" if object_3d is not None else "no target detected"
        self._emit_frame("Initial VLM Plan", plan, object_3d, obj_msg, pause=True)

        action_buffer = list(plan["actions"])
        action_count = 0
        replan_streak = 0            # consecutive re-plan failures (capped to avoid hang)

        while self._step < max_st:
            if self._is_success():
                _succ_step = self._step          # true success step (before post-success hold)
                self._hold_and_capture(20)
                return TrialResult(True, _succ_step, None, plan)

            # Held latch: once the cube is observed clearly off the table with the gripper closed
            # (a grasp + lift succeeded), mark it held for the rest of the trial. This SURVIVES a
            # later descend-to-place (the cube is intentionally lowered again) so the planner/query
            # re-check is never fooled into thinking the grasp failed mid-place and re-targeting the
            # already-held object. Cleared only by an actual release or a trial reset.
            if self._gripper_closed and getattr(self.env, "cube_body_id", None) is not None:
                obj_z_held = float(self.env.sim.data.body_xpos[self.env.cube_body_id][2])
                if obj_z_held > self._tz + 0.06:
                    self._object_held = True

            if len(action_buffer) == 0:
                # Capture the current state once for this re-plan round (shared by query
                # re-check, stereo re-detection, and the VLM re-plan below).
                ee = self._ee_pos()
                img_L, img_R = self._all_views()
                grip_state = "closed" if self._gripper_closed else "open"
                # Report the TRUE held status, not the raw cube height. A descend-to-place
                # intentionally lowers a HELD cube back near the table, which used to read as
                # "still on table - grasp failed" and made the query re-check flip back to
                # re-locating the cube (the oscillation/timeout cause). The self._object_held
                # latch (set by a proven grasp+lift, cleared only by release) is authoritative:
                # if it is True the object IS secured regardless of the current cube z.
                if self._gripper_closed:
                    cube_id = getattr(self.env, "cube_body_id", None)
                    if self._object_held:
                        grip_state += (" (object SECURED — grasp succeeded and the object is held; "
                                       "do NOT re-grasp it; proceed to place/next step)")
                    elif cube_id is not None:
                        obj_z = float(self.env.sim.data.body_xpos[cube_id][2])
                        grip_state += (f" (gripper closed but object NOT confirmed held, cube at "
                                       f"z={obj_z:.3f} — the grasp may have missed; locate the object "
                                       f"to grasp it)")
                history_images = {}
                for i, (frames_dict, _) in enumerate(self._history_frames):
                    for cam_name, frame in frames_dict.items():
                        history_images[f"history_{i}_{cam_name}"] = frame
                history_desc = self._build_history_desc()

                # Re-detect (only if this task needs an object) and re-plan.
                if self._needs_object:
                    # Per-round query regeneration: when no fixed query is configured
                    # (bbox_target is None), ask the VLM which single object to locate next
                    # from the current state. A fixed base_cfg.bbox_target skips regeneration.
                    if base_cfg.bbox_target is None:
                        cur_query = self.planner.regenerate_query(
                            img_L, img_R, task, self._step, completed_actions, ee,
                            gripper_state=grip_state,
                            history_images=history_images, history_desc=history_desc)
                        if cur_query:
                            self._target_desc = cur_query
                    new_object_3d, new_bboxes = self._detect_and_project(task, self._target_desc)
                    if new_object_3d is not None:
                        object_3d = new_object_3d
                    else:
                        # Fall back to THIS query's last-known pose (per-query cache) — never a
                        # different object's stale coordinates. If this query was never detected,
                        # there is genuinely no target this round and the planner must wait/move_by.
                        cached = self._target_cache.get(self._target_desc)
                        if cached is not None:
                            object_3d = np.array(cached, dtype=float)
                            print(f"  [WARN] re-detect failed, using cached '{self._target_desc}' "
                                  f"at ({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})", flush=True)
                        else:
                            object_3d = None
                            print(f"  [WARN] re-detect failed, no cached position for '{self._target_desc}'", flush=True)

                _t0 = time.time()

                new_plan = self.planner.plan_closedloop(
                    img_L, img_R, task, self._step, completed_actions, ee, n,
                    object_3d, gripper_state=grip_state,
                    history_images=history_images, history_desc=history_desc,
                    needs_object=self._needs_object)
                _elapsed = time.time() - _t0

                if new_plan is not None and len(new_plan["actions"]) > 0:
                    print(f"    Got {len(new_plan['actions'])} actions in {_elapsed:.0f}s", flush=True)
                    action_buffer = list(new_plan["actions"])
                    action_count = 0
                    replan_streak = 0
                    self._emit_frame("Re-plan", new_plan, object_3d,
                                   f"{len(action_buffer)} actions ({_elapsed:.0f}s)", pause=True)
                else:
                    replan_streak += 1
                    if replan_streak >= 10:
                        print(f"  VLM re-plan failed {replan_streak}x in a row, giving up", flush=True)
                        return TrialResult(False, self._step, "replan_failed", plan)
                    print(f"  VLM re-plan failed ({_elapsed:.0f}s), retrying... ({replan_streak}/10)", flush=True)
                    continue

            # Execute next action
            action = action_buffer.pop(0)
            action_count += 1
            atype = action["type"]
            params = {k: v for k, v in action.items() if k != "type"}
            self._action_count = action_count

            self._emit_frame(f"Move #{action_count}: {atype}",
                             t3d=object_3d, msg=f"params={json.dumps(params)}")

            result_str = self._execute_action(action, object_3d)
            self._record_history(action, result_str)
            completed_actions.append(f"{atype}(step={self._step},{result_str})")
            if len(completed_actions) > 20:
                completed_actions = completed_actions[-20:]

        return TrialResult(self._is_success(), self._step,
                           "timeout" if not self._is_success() else None, plan)

---
## 12. Base-Aligned Evaluation and Video Recording

In [25]:
def build_base_planner_for_backbone(backbone: QwenLoRABackbone) -> VLMPlanner:
    sync_base_inference_cfg_from_rl(cfg, task=cfg.task_prompt, seed=cfg.seed)
    return VLMPlanner(
        base_cfg.vlm_model_id,
        base_cfg.use_4bit,
        model=backbone.model,
        processor=backbone.processor,
    )


def summarize_inference_results(results, label: str):
    n = len(results)
    ok = sum(1 for r in results if r.success)
    errs = {}
    for r in results:
        k = r.error or "ok"
        errs[k] = errs.get(k, 0) + 1
    rate = (ok / n) if n else 0.0
    print(f"\n{'=' * 40}\n{label}: {ok}/{n} = {100 * rate:.1f}%")
    for e, c in sorted(errs.items()):
        print(f"  {e}: {c}")
    print('=' * 40)
    return {
        "success_rate": rate,
        "successes": ok,
        "trials": n,
        "errors": errs,
        "results": results,
    }


def evaluate_base_inference_trials(
    cfg: RLConfig,
    planner: VLMPlanner,
    task: Optional[str] = None,
    n_trials: Optional[int] = None,
    label: str = "Base-Aligned Inference",
):
    task = sync_base_inference_cfg_from_rl(cfg, task=task or cfg.task_prompt, seed=cfg.seed)
    if hasattr(planner, "model"):
        planner.model.eval()
    planner._call_count = 0
    if hasattr(planner, "_trace_enabled"):
        planner._trace_enabled = False
        planner._trace_logprobs = []
    n_trials = cfg.inference_eval_trials if n_trials is None else n_trials

    print(f"\nRunning {label} over {n_trials} trial(s) with base-style prompts/controller...")
    results = []
    for trial in range(n_trials):
        trial_seed = cfg.seed + trial
        base_cfg.seed = trial_seed
        set_base_inference_seed(trial_seed)
        env = make_base_inference_env(base_cfg, seed=trial_seed)
        try:
            ctrl = VLMController(env, planner, on_frame=None)
            result = ctrl.run_closedloop(task)
            results.append(result)
            tag = "OK" if result.success else "FAIL"
            print(f"  [{tag}] trial={trial + 1}/{n_trials} seed={trial_seed} steps={result.steps} error={result.error or 'ok'}")
        finally:
            env.close()

    base_cfg.seed = cfg.seed
    set_base_inference_seed(cfg.seed)
    return summarize_inference_results(results, label)


def evaluate_policy(
    cfg: RLConfig,
    backbone: Optional[QwenLoRABackbone] = None,
    seed: Optional[int] = None,
    video_path: Optional[str] = None,
):
    """Inference uses the exact planner / prompt / detector / controller stack copied
    from robot_vlm_base.ipynb. If a trained backbone is provided, the planner reuses its
    current LoRA-adapted model weights."""
    task = sync_base_inference_cfg_from_rl(cfg, task=cfg.task_prompt, seed=seed)
    planner = build_base_planner_for_backbone(backbone) if backbone is not None else get_base_planner()
    if hasattr(planner, "model"):
        planner.model.eval()

    if video_path is not None:
        env = make_base_inference_env(base_cfg, seed=base_cfg.seed)
        cam = VideoCapture(fps=cfg.video_fps)
        try:
            img_L = render_rgb(env, camera=base_cfg.left_camera)
            img_R = render_rgb(env, camera=base_cfg.right_camera)
            cam.snap(img_L, img_R, "Initial", None, None, f'Task: "{task}"')
            ctrl = VLMController(env, planner, on_frame=cam.snap)
            result = ctrl.run_closedloop(task)
            final_msg = "SUCCESS" if result.success else f"FAILED ({result.error})"
            img_L = render_rgb(env, camera=base_cfg.left_camera)
            img_R = render_rgb(env, camera=base_cfg.right_camera)
            cam.snap(img_L, img_R, final_msg, result.plan, None, f"Steps: {result.steps}", step=result.steps)
            saved_path = cam.save(video_path)
            return {
                "success": result.success,
                "steps": result.steps,
                "error": result.error,
                "video_path": saved_path,
            }
        finally:
            env.close()

    env = make_base_inference_env(base_cfg, seed=base_cfg.seed)
    try:
        ctrl = VLMController(env, planner, on_frame=None)
        result = ctrl.run_closedloop(task)
        return {
            "success": result.success,
            "steps": result.steps,
            "error": result.error,
            "video_path": None,
        }
    finally:
        env.close()

# Example after training:
# eval_result = evaluate_policy(cfg, train_state["backbone"], video_path="qwen_base_style_eval.mp4")
# print(eval_result)


---
## 13. Build Objects and Train

In [26]:
def build_training_objects(cfg: RLConfig):
    backbone = QwenLoRABackbone(cfg)
    trainable_backbone = [p for p in backbone.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(
        [
            {"params": trainable_backbone, "lr": cfg.backbone_lr},
        ]
    )

    print(f"Qwen hidden size: {backbone.hidden_size}")
    print(f"Backbone trainable params: {count_trainable_parameters(backbone):,}")
    print(f"Backbone total params: {count_all_parameters(backbone):,}")
    print("Training mode: planner-only GRPO-style RL (no policy head)")
    return backbone, optimizer


def save_checkpoint(cfg: RLConfig, backbone: QwenLoRABackbone, optimizer, history, update_idx: int):
    os.makedirs(cfg.checkpoint_dir, exist_ok=True)
    ckpt_root = os.path.join(cfg.checkpoint_dir, f"{cfg.run_name}_update{update_idx:04d}")
    adapter_dir = os.path.join(ckpt_root, "qwen_lora_adapter")
    os.makedirs(ckpt_root, exist_ok=True)
    backbone.model.save_pretrained(adapter_dir)
    backbone.processor.save_pretrained(adapter_dir)
    torch.save(
        {
            "config": cfg.__dict__,
            "update": update_idx,
            "optimizer": optimizer.state_dict(),
            "history": history,
        },
        os.path.join(ckpt_root, "rl_state.pt"),
    )
    return ckpt_root


def compute_planner_trial_reward(ctrl: VLMController, result: TrialResult, cfg: RLConfig) -> float:
    if cfg.grpo_use_binary_success_reward:
        return 1.0 if result.success else 0.0

    reward = cfg.planner_reward_success_bonus if result.success else 0.0
    env = ctrl.env
    cube_id = getattr(env, "cube_body_id", None)
    if cube_id is not None:
        cube_pos = np.asarray(env.sim.data.body_xpos[cube_id], dtype=float)
        table_z = float(env.table_offset[2]) if hasattr(env, "table_offset") else ctrl._tz
        lift = float(np.clip((cube_pos[2] - table_z) / 0.15, 0.0, 1.0))
        reward += cfg.planner_reward_lift_bonus * lift
        if getattr(env, "bin_body_id", None) is not None:
            bin_pos = np.asarray(env.sim.data.body_xpos[env.bin_body_id], dtype=float)
            xy_dist = float(np.linalg.norm(cube_pos[:2] - bin_pos[:2]))
            bin_term = float(np.exp(-5.0 * xy_dist))
            reward += cfg.planner_reward_bin_bonus * bin_term
            reward += cfg.planner_reward_release_bonus * float(not ctrl._gripper_closed)
    reward -= cfg.planner_reward_step_penalty * float(result.steps)
    return float(reward)


def collect_grpo_group(cfg: RLConfig, backbone: QwenLoRABackbone, update_idx: int):
    task = sync_base_inference_cfg_from_rl(cfg, task=cfg.task_prompt, seed=cfg.seed)
    group_seed = cfg.seed + update_idx
    base_cfg.seed = group_seed

    candidates = []
    for sample_idx in range(cfg.grpo_group_size):
        planner = build_base_planner_for_backbone(backbone)
        planner_seed = group_seed * 1000 + sample_idx
        set_base_inference_seed(planner_seed)
        env = make_base_inference_env(base_cfg, seed=group_seed)
        planner.begin_trace()
        try:
            ctrl = VLMController(env, planner, on_frame=None)
            result = ctrl.run_closedloop(task)
            logprob = planner.end_trace()
            reward = compute_planner_trial_reward(ctrl, result, cfg)
            candidates.append(
                {
                    "sample_idx": sample_idx,
                    "seed": group_seed,
                    "planner_seed": planner_seed,
                    "result": result,
                    "reward": reward,
                    "logprob": logprob,
                }
            )
        finally:
            env.close()

    base_cfg.seed = cfg.seed
    set_base_inference_seed(cfg.seed)
    return candidates


def grpo_update(backbone: QwenLoRABackbone, optimizer, candidates, cfg: RLConfig):
    valid = [c for c in candidates if c["logprob"] is not None]
    if not valid:
        success_rate = float(np.mean([1.0 if c["result"].success else 0.0 for c in candidates])) if candidates else 0.0
        reward_mean = float(np.mean([c["reward"] for c in candidates])) if candidates else float("nan")
        return {"loss": float("nan"), "reward_mean": reward_mean, "success_rate": success_rate}

    rewards = torch.tensor([c["reward"] for c in valid], dtype=torch.float32, device=DEVICE)
    advantages = normalize_rewards(rewards)
    losses = [-(adv * cand["logprob"]) for adv, cand in zip(advantages, valid)]
    loss = torch.stack(losses).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(
        (p for p in backbone.parameters() if p.requires_grad),
        cfg.max_grad_norm,
    )
    optimizer.step()

    success_rate = float(np.mean([1.0 if c["result"].success else 0.0 for c in candidates])) if candidates else 0.0
    reward_mean = float(np.mean([c["reward"] for c in candidates])) if candidates else float("nan")
    return {"loss": float(loss.item()), "reward_mean": reward_mean, "success_rate": success_rate}


def train_lora_grpo(cfg: RLConfig):
    set_seed(cfg.seed)
    backbone, optimizer = build_training_objects(cfg)
    history = []
    t0 = time.time()
    pre_eval = None
    post_eval = None
    eval_planner = None

    if cfg.run_pre_post_inference_eval:
        print("\n[Eval] Base-style inference BEFORE RL training")
        eval_planner = build_base_planner_for_backbone(backbone)
        pre_eval = evaluate_base_inference_trials(
            cfg,
            eval_planner,
            n_trials=cfg.inference_eval_trials,
            label="Before RL",
        )

    print("\n[Train] GRPO-style planner RL: same-scene grouped candidates, reward from task success")

    for update in range(1, cfg.total_updates + 1):
        candidates = collect_grpo_group(cfg, backbone, update_idx=update - 1)
        update_stats = grpo_update(backbone, optimizer, candidates, cfg)
        mean_steps = float(np.mean([c["result"].steps for c in candidates])) if candidates else float("nan")
        row = {
            "update": update,
            "reward_mean": update_stats["reward_mean"],
            "success_rate": update_stats["success_rate"],
            "loss": update_stats["loss"],
            "episode_length_mean": mean_steps,
            "elapsed_sec": time.time() - t0,
        }
        history.append(row)
        print(
            f"update={update:04d} "
            f"reward={row['reward_mean']:.4f} "
            f"success={100.0 * row['success_rate']:.1f}% "
            f"len={row['episode_length_mean']:.2f} "
            f"loss={row['loss']:.4f}"
        )

        if update % cfg.save_every == 0 or update == cfg.total_updates:
            ckpt = save_checkpoint(cfg, backbone, optimizer, history, update)
            print(f"saved checkpoint: {ckpt}")

    if cfg.run_pre_post_inference_eval:
        print("\n[Eval] Base-style inference AFTER RL training")
        post_eval = evaluate_base_inference_trials(
            cfg,
            eval_planner if eval_planner is not None else build_base_planner_for_backbone(backbone),
            n_trials=cfg.inference_eval_trials,
            label="After RL",
        )

    return {
        "config": cfg,
        "backbone": backbone,
        "optimizer": optimizer,
        "history": history,
        "pre_eval": pre_eval,
        "post_eval": post_eval,
    }


# Start training when you are ready.
train_state = train_lora_grpo(cfg)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)



Before RL: 4/5 = 80.0%
  ok: 4
  timeout: 1

[Train] GRPO-style planner RL: same-scene grouped candidates, reward from task success
Loading Qwen3-VL: Qwen/Qwen3-VL-4B-Instruct ...
  VLM loaded (existing backbone)


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

--- VLM Call #1 ---
  [TASK ANALYSIS] needs_detection=True, target='red cube'
  [DETECT] using target='red cube'
  [OWLv2] L=[0.4877106249332428, 0.2304767370223999, 0.5638346076011658, 0.32259660959243774] (score=0.691)
  [OWLv2] R=[0.7299588322639465, 0.31048503518104553, 0.8127846121788025, 0.41017165780067444] (score=0.658)
  [OWLv2] 3D target=(-0.193, -0.118, 0.821)


OutOfMemoryError: CUDA out of memory. Tried to allocate 798.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 506.81 MiB is free. Including non-PyTorch memory, this process has 14.06 GiB memory in use. Of the allocated memory 13.06 GiB is allocated by PyTorch, and 893.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## Notes

Current design summary:
- Qwen itself is the trainable policy via LoRA.
- Training uses natural-language / DSL plan generation, not a separate continuous-action policy head.
- The RL loop is GRPO-style in spirit: same-scene grouped plan samples, task-driven reward, relative update.

Important tradeoff:
- This is much heavier than small-policy RL because every update still depends on repeated Qwen generation and execution.

Recommended next extensions if you continue:
1. Replace post-hoc response scoring with true sampled token logprob tracking during generation.
2. Add multi-scene batching or parallel workers to reduce variance and improve throughput.
3. Tighten reward design if binary success is too sparse.
4. Add explicit held-out evaluation seeds for cleaner before/after comparison.
5. If memory becomes tight, narrow `lora_target_modules` or reduce image resolution.
